<!-- notebook-header -->
<a id="top"></a>

# 7. Heaps, Intervals & Greedy

*LeetCode Playbook · notebook 7 of 12* · Always the cheapest next, overlapping ranges, and local choices you can prove are safe.

**In this notebook:** [Heaps](#s13) · [Intervals & Sweep Line](#s14) · [Greedy](#s15)

← [Trees & Tries](06_Trees_Tries.ipynb) · [All notebooks](01_Start_Here.ipynb#notebooks) · [A-Z problem finder](12_Interview_Day.ipynb#finder) · [Backtracking (and a short DP map)](08_Backtracking.ipynb) →

Run the setup cell below first; after that every section runs from its first cell.

In [ ]:
# Setup: everything this notebook imports. Run this cell first.
from collections import Counter, defaultdict, deque, OrderedDict
from functools import cmp_to_key, lru_cache
from itertools import accumulate, combinations, permutations, product
import bisect, heapq, math, random, string

<a id="s13"></a>

## Heaps

> A heap is a pile that always hands you the smallest item first. It keeps only a *partial* order, every parent at most its children, which is exactly enough to read the minimum in O(1) and to add or remove an item in O(log n).

**Reach for it when** you need the smallest or largest item *again and again* while items keep arriving or leaving: **top k, k-th largest, k closest**, **merge k sorted** lists, the **median of a stream**, "always process the cheapest / earliest / most frequent next", scheduling with deadlines or cooldowns, or growing a frontier from its lowest point, which is the shape of Dijkstra's algorithm in [Graphs III](09_Graphs.ipynb#s19).

**In this repo:** `heap/` (20 problems) · bank: `practice/simple/32_k_closest_points_to_origin.py`, `practice/simple/33_task_scheduler.py`, `practice/simple/34_find_median_from_data_stream.py`, `practice/simple/35_merge_k_sorted_lists.py` · basics: `practice/simple/basics/heaps/` (`01_heapify_by_hand.py`, `02_push_and_pop_by_hand.py`, `03_top_k_with_size_k_min_heap.py`, `04_max_heap_by_negation_and_tuples.py`, `05_kth_largest_in_a_stream.py`, `06_merge_k_sorted_arrays.py`), `practice/simple/basics/sorting/04_heap_sort.py`

### The picture

```text
the list:  [1, 3, 2, 7, 4, 9, 5]       node i has children 2i+1 and 2i+2, parent (i-1)//2

the tree:          1                   the only rule: every parent <= its children
                 /   \                 -> the smallest is always heap[0]
                3     2                -> nothing else is in order: heap[1] = 3 is not the
               / \   / \                  second smallest (2 is), heap[-1] = 5 is not the
              7   4 9   5                 largest (9 is)

push 0:  append it as the last leaf, then swap it UP while its parent is bigger
         [1, 3, 2, 7, 4, 9, 5, 0]  ->  0 passes 7, 3, 1  ->  [0, 1, 2, 3, 4, 9, 5, 7]
pop:     take heap[0], move the last leaf to the root, swap it DOWN toward its smaller child
         both walk a single root-to-leaf path: at most log2(n) swaps
```

Ask a plain list for its smallest item and it scans everything, O(n), every time you ask. Sort the list first and you pay O(n log n) for an order you mostly never use. A heap keeps just enough order to answer one question, "what is the smallest?", and repairs that order along a single root-to-leaf path after each change: O(1) to peek, O(log n) to push or pop.

Capped at k items, a heap answers "the top k of n" in O(n log k) time and O(k) memory, even on a stream too big to store.

`heapify` builds a heap from a whole list in O(n), not O(n log n). It sifts each parent *down*, and a node can only sink as far as its height: half the nodes are leaves that never move, a quarter sink at most one level, an eighth at most two, and that sum stays below n.

Python's `heapq` keeps a min-heap in a plain list, and [Python Toolkit](01_Start_Here.ipynb#s02) already shows its everyday calls: push, pop, `heapify`, `nlargest`, negation for a max-heap, and a counter in the tuple so that a tie never compares the payload. Three more calls earn their place in this section. `heappushpop` pushes and then pops in a single sift, `heapreplace` pops first and then pushes, and `merge` is a ready-made, lazy k-way merge of sorted lists, the second template below.

In [ ]:
h = [2, 5, 8]
print(heapq.heappushpop(h, 0), h)                        # 0 [2, 5, 8]  push, then pop
h = [2, 5, 8]
print(heapq.heapreplace(h, 0), h)                        # 2 [0, 5, 8]  pop, then push
print(list(heapq.merge([1, 4, 5], [1, 3, 4], [2, 6])))   # [1, 1, 2, 3, 4, 4, 5, 6]

**Try it**
- Run `heapq.heappushpop(h, 9)` on a fresh `h = [2, 5, 8]`: it returns 2 and leaves `[5, 9, 8]`. The newcomer stays and the weakest leaves, which is the keep-k template below in one call.
- Call `heapq.heapreplace([], 1)`: `IndexError`, because it pops before it pushes. `heapq.heappushpop([], 1)` returns 1, because it pushes first.
- Feed `merge` a list that is not sorted: `list(heapq.merge([3, 1], [2]))` gives `[2, 3, 1]`. It trusts every input to be sorted and never checks.
- Replace `list(...)` with `next(...)` in the last line: it prints 1. `merge` hands out one item per request, so it also merges streams too long to hold.

### From idea to code

*Keep the candidates in a heap keyed by "who should go next"; pop the best, deal with it, push any new candidates it creates; and when only the best k matter, let the heap throw out its weakest member whenever it holds k + 1.*

Start with the question the k *largest* raise: why a *min*-heap? Each newcomer asks one thing, "am I better than the weakest of the k?" The weakest of the k largest is their minimum, so the minimum must sit on top, and a min-heap puts it there. A max-heap of all n items would also work, popped k times, but it holds all n: O(n) memory, and useless on a stream. When the largest must sit on top instead, push `-key` and read `-heap[0]`.

Two templates cover the whole section. The first keeps the k best, as in Kth Largest Element in an Array, the k-th largest value of a list, in Kth Largest Element in a Stream, the same after every `add`, and in K Closest Points to Origin, the k points nearest the origin.

Its **State** is a min-heap of the k largest items seen so far, and its **Definition** gives the root its meaning: `heap[0]` is the weakest of them. The **Invariant**, true after every step, is that the heap holds exactly the k largest items seen so far, or all of them while fewer than k have arrived. A **Step** pushes the newcomer; if that makes k + 1, the **Fix** pops the weakest of them, which may be the newcomer itself.

The **Record** is the root: after the fix, `heap[0]` is the k-th largest so far, which is exactly what Kth Largest Element in a Stream returns at the end of each `add`. The **Init** is an empty heap. The **Return** is the root, or the heap's items, sorted first when order matters, because a heap keeps only a partial order.

The second template takes the next item from a frontier, as in Merge k Sorted Lists, which merges k sorted lists into one. Its state is one candidate per source, for a merge the tuple `(value, list i, index j)`, so that the root is the next item overall. The invariant is that every item not yet popped is in the heap or waits behind one in its own list.

A step pops the root and records it at once, because pops come out in sorted order. The fix pushes the popped item's successor, the next item of the same list, and the invariant holds again. Init is `heapify` on one head per list, in O(n). The return is the sequence of pops, or the value recorded when the target shows up; a pop needed from an empty heap means the input was impossible, −1 or `""`.

The same loop drives two Hard problems at the end of the section: one finger per list finds the smallest range that covers k lists, and a wall that starts at the border measures the water a height map traps. The flood records earlier, at the push, because a cell's level is final the moment it is first reached.

The order of the two lines is a decision too. Push first, then evict: the heap compares the newcomer with the weakest of the k for you, and if the newcomer is the weakest of the k + 1, it is the one that leaves. Evict first and you throw out a kept item before anyone has checked that the newcomer is better. In the merge the order is pop, record, refill, because only after the pop do you know which list needs a new head.

The cell below holds both templates. `k_largest` returns the k largest numbers of a list, largest first: `[3, 1, 5, 12, 2, 11]` with k = 3 gives `[12, 11, 5]`. `merge_sorted` is a k-way merge, which merges k sorted lists into one sorted list by always taking the smallest head: `[[1, 4, 5], [1, 3, 4], [2, 6]]` becomes `[1, 1, 2, 3, 4, 4, 5, 6]`. The list index sits in the middle of each tuple so that equal values never compare the payload.

In [ ]:
def k_largest(nums, k):
    heap = []                                # STATE + INIT: min-heap of the k largest so far; heap[0] = the weakest
    for x in nums:
        heapq.heappush(heap, x)              # STEP: x joins the candidates
        if len(heap) > k:                    # FIX: k + 1 candidates, the weakest leaves (maybe x itself)
            heapq.heappop(heap)
    return sorted(heap, reverse=True)        # RETURN: the k largest, largest first


def merge_sorted(lists):
    heap = [(lst[0], i, 0) for i, lst in enumerate(lists) if lst]   # STATE + INIT: (value, list i, index j), one head per list
    heapq.heapify(heap)
    out = []
    while heap:
        val, i, j = heapq.heappop(heap)      # STEP: the smallest head is the next item overall
        out.append(val)                      # RECORD: pops come out in sorted order
        if j + 1 < len(lists[i]):            # FIX: list i lost its head; its next item takes the seat
            heapq.heappush(heap, (lists[i][j + 1], i, j + 1))
    return out                               # RETURN


print(k_largest([3, 1, 5, 12, 2, 11], 3))            # [12, 11, 5]
print(merge_sorted([[1, 4, 5], [1, 3, 4], [2, 6]]))  # [1, 1, 2, 3, 4, 4, 5, 6]

**Try it**
- Change `if len(heap) > k:` to `>=` and rerun: `[12, 11]`. The heap is now capped at k − 1.
- Evict first: replace the two lines in the loop with `if len(heap) == k: heapq.heappop(heap)` followed by `heapq.heappush(heap, x)`. `k_largest([5, 1], 1)` returns `[1]`: the 5 was thrown out before anyone checked that 1 is worse.
- Print `heap` at the top of the `while` loop in `merge_sorted`: it never holds more than 3 entries, one per list. That is where O(N log k) comes from.
- Remove `if lst` from the first line of `merge_sorted` and call `merge_sorted([[], [2]])`: `IndexError`, because an empty list has no head.

### Watch it work

The trace runs the template's own two lines, push then evict, and says who left at each step. Once the heap holds k items its root is a doorman: a newcomer stays only if it beats the root, and then the root leaves. Watch `[3, 1, 5, 12, 2, 11]` with k = 3 fill up for three steps and then bounce or admit each newcomer.

In [ ]:
def trace_k_largest(nums, k):
    heap = []
    for x in nums:
        heapq.heappush(heap, x)                                   # STEP
        gone = heapq.heappop(heap) if len(heap) > k else None     # FIX
        note = ("filling up" if gone is None else
                "bounced: x was the weakest" if gone == x else f"{gone} leaves")
        print(f"x={x:<3} heap={str(heap):<12} root={heap[0]:<3} {note}")


trace_k_largest([3, 1, 5, 12, 2, 11], 3)

**Try it**
- Run it with `k = 1`: the heap is a single "best so far", and the root only climbs (3, then 5, then 12).
- Run `[1, 2, 3, 4, 5]` and then `[5, 4, 3, 2, 1]` with `k = 2`: once the heap is full, every newcomer stays on the rising list and every newcomer bounces on the falling one. Same O(n log k) bound, very different traffic.
- Rewrite the FIX compare-first: `if len(heap) < k: heapq.heappush(heap, x)` / `elif x > heap[0]: heapq.heapreplace(heap, x)`. The `root` column is the same at every step, but a newcomer that can't beat the root never enters. With `>=` in place of `>`, `[5, 5, 5, 5]` and k = 2 swap equal values in and out for nothing.
- Read the `heap=` column: it need not be sorted (`[3, 12, 5]`), yet once the heap is full, `root` is always the k-th largest so far.

### Where it goes wrong

The heap's rules are few, so nearly every bug is one of these eight.

1. **Ties compare the payload.** `(dist, node)` raises `TypeError` the first time two distances are equal and the nodes can't be compared: `heapq.heappush([(1, {"id": 1})], (1, {"id": 2}))` crashes on the two dicts, as the two `Job`s crash in [Python Toolkit](01_Start_Here.ipynb#s02). Push `(dist, i, node)` with a unique `i`, the list index or a running counter.
2. **Negating only half of the time.** Push `-x`, read `-heap[0]`, negate what you pop. In the median finder below, dropping the minus in step 2 (`heappush(self.high, heappop(self.low))`) makes `median()` return −5.0 after `add(5)`. A missing minus gives a silently wrong answer, never an error.
3. **The wrong heap for top-k.** k largest → *min*-heap capped at k; k smallest or k closest → *max*-heap capped at k. The other way round evicts your best items: `[3, 1, 5]`, k = 1, with a max-heap keeps 1.
4. **Changing a key that is already in the heap.** The heap never notices: `h = [[1, "a"], [2, "b"]]; h[1][0] = 0; heapq.heappop(h)` returns `[1, "a"]`. Push a fresh entry instead and skip the stale one when it surfaces: that is lazy deletion, which asks right after each pop whether the entry is still valid.
5. **Popping an empty heap.** `IndexError`. IPO, which picks the most profitable projects the capital can afford, crashes below on `max_capital(1, 0, [5], [1])` without its `if not heap: break`. An empty heap is often the "impossible" answer.
6. **`heappush` onto a list that is not a heap.** `heapq` never checks: `h = [5, 1]; heapq.heappush(h, 3)` leaves `h[0] == 3`, not 1. Call `heapify` once first.
7. **Using what `heapify` returns.** It works in place and returns `None`: `h = heapq.heapify([3, 1]); h[0]` raises `TypeError: 'NoneType' object is not subscriptable`.
8. **Two successors per pop.** Find K Pairs with Smallest Sums (373), the k pairs with the smallest sums from two sorted lists, and Kth Smallest Element in a Sorted Matrix (378), whose rows and columns are sorted, both walk a grid of sorted values. Pushing both `(i + 1, j)` and `(i, j + 1)` after every pop reaches the same cell twice: with `nums1 = nums2 = [1, 2]` and k = 5, the pair `[2, 2]` comes out twice. Seed one head per row and only move right, or keep a `seen` set.

### Edge cases to say out loud

k = 0 · k ≥ n · duplicates (they count separately) · empty lists among the k lists · all lists empty · equal keys with uncomparable payloads · negatives under negation · an even count for the median (average, as a float) · the heap running dry before you are done (impossible input). The cell checks the cases that apply to the two templates.

In [ ]:
assert k_largest([5, 5, 5], 2) == [5, 5]                  # duplicates count separately
assert k_largest([2, 1], 5) == [2, 1]                     # k >= n: keep everything
assert k_largest([4, 1], 0) == []                         # k = 0: everything leaves again
assert k_largest([], 3) == []
assert k_largest([-1, -7, -3], 2) == [-1, -3]
assert merge_sorted([]) == [] and merge_sorted([[], []]) == []
assert merge_sorted([[1, 1], [1]]) == [1, 1, 1]           # equal heads: the list index breaks the tie
assert merge_sorted([[-3, 0], [-5], [7]]) == [-5, -3, 0, 7]
print("edge cases pass")

**Try it**
- Predict, then add: `assert k_largest([2, 9, 4], 3) == [9, 4, 2]` (k = n gives the input sorted, largest first).
- Why does `merge_sorted([[1, 1], [1]])` never compare anything but numbers? Print the heap after `heapify`: `[(1, 0, 0), (1, 1, 0)]`; the second field already differs.
- Merge k Sorted Lists (23) passes linked-list nodes: push `(node.val, i, node)` and refill with `node.next`. Drop the `i` and merge two lists whose heads are equal: `TypeError`, because Python then compares two `ListNode`s.

### Variations

Every variation keeps one of the two loops, push-and-evict or pop-and-refill, and changes one thing: the key, the number of heaps, or what waits beside the heap.

| Variation | What changes from the template | Problems |
|---|---|---|
| **Keep the k best** | min-heap capped at k for the k largest; max-heap, negated, for the k smallest or closest; Maximum Performance of a Team, which picks at most k engineers to maximise speed sum × minimum efficiency, sorts by efficiency and keeps the k best speeds | Kth Largest Element in an Array (215), Kth Largest Element in a Stream (703), K Closest Points to Origin (973), Maximum Performance of a Team (1383) |
| **Custom order** | strings can't be negated: a class with `__lt__` where "less" means "worse", or heapify all `(-count, word)` and pop k | Top K Frequent Words (692): the k most frequent words, ties alphabetical |
| **Two heaps** | max-heap for the low half, min-heap for the high half; the median sits at the roots | Find Median from Data Stream (295): the median after every new number; Sliding Window Median (480): the median of every window of k numbers, in [Sliding Window](03_Two_Pointers_Sliding_Window_Strings.ipynb#s06) |
| **Cooldown** | max-heap of counts plus a FIFO queue, or one held item, of what can't be used yet; Rearrange String k Distance Apart wants equal letters at least k apart, so its queue has length k | Task Scheduler (621): the least time to run tasks with a cooldown; Reorganize String (767): no two equal letters side by side; Rearrange String k Distance Apart (358) |
| **K-way merge** | the template: one head per sorted list, pop the smallest, push its successor; Design Twitter's news feed merges the followed users' tweet lists, newest first, and stops after 10 pops; K-th Smallest Prime Fraction merges the sorted rows of fractions arr[i] / arr[j] | Merge k Sorted Lists (23), Design Twitter (355), K-th Smallest Prime Fraction (786) |
| **Two orders at once** | Meeting Rooms III gives each meeting the lowest free room, or delays it until one frees up: one heap of free rooms by id, one of busy rooms by end time | Meeting Rooms III (2402) |
| **Shrink the max** | Minimize Deviation in Array may halve evens and double odds to shrink max − min: a max-heap, since only lowering the max can shrink the gap, with the min tracked beside it | Minimize Deviation in Array (1675) |
| *Second pass:* **unlock, then take the best** | sort by the unlock key, push everything unlocked, pop the best when you must choose | IPO (502); Minimum Number of Refueling Stops (871): the fewest stops on the way to a target; Single-Threaded CPU (1834): the order in which one CPU runs its tasks, always the shortest one that has arrived |
| *Second pass:* **take now, regret later** | take every item; when a limit breaks, pop the worst item you took | Course Schedule III (630): the most courses that meet their deadlines; Furthest Building You Can Reach (1642): how far a fixed stock of bricks and ladders carries you up a row of buildings |
| *Second pass:* **lazy deletion** | leave dead entries in; pop them only when they reach the top | The Skyline Problem (218): the outline of a row of buildings; Sliding Window Median (480); Minimum Interval to Include Each Query (1851): the smallest interval holding each query point, in [Intervals & Sweep Line](#s14) |
| *Second pass:* **lowest frontier first** | heap of frontier cells keyed by level or distance, the Dijkstra shape | Trapping Rain Water II (407): the water a 2-D height map holds; Network Delay Time (743): how long a signal takes to reach every node, in [Graphs III](09_Graphs.ipynb#s19) |
| *Second pass:* **merge + running max** | the range `[heap min, running max]` covers every list; advance the min | Smallest Range Covering Elements from K Lists (632): the shortest range holding a number of every list |

The template kept the k largest; the k *closest* are its mirror image. K Closest Points to Origin asks for the k points nearest the origin, so `[[1, 3], [-2, 2]]` with k = 1 gives `[[-2, 2]]`. Cap a *max*-heap at k: its root is the farthest point you kept, and a newcomer stays only if it is closer. Squared distances keep the order of distances, so no square root is needed.

Top K Frequent Words asks for the k most frequent words, ties broken alphabetically: `["i", "love", "leetcode", "i", "love", "coding"]` with k = 2 gives `["i", "love"]`. Words cannot be negated, so each entry gets a `__lt__` in which "less" means "worse", fewer copies or a later word, and the same push-and-evict loop works. When k is small it is simpler to heapify all `(-count, word)` pairs and pop k times, O(m + k log m) for m distinct words.

In [ ]:
def k_closest(points, k):
    heap = []                                # STATE: max-heap via (-dist, x, y); root = the farthest kept
    for x, y in points:
        heapq.heappush(heap, (-(x * x + y * y), x, y))   # STEP: squared distance, same order, no sqrt
        if len(heap) > k:                    # FIX: the farthest of k + 1 leaves
            heapq.heappop(heap)
    return sorted([x, y] for _, x, y in heap)            # RETURN (sorted only for a stable printout)


class Entry:                                 # "less" means "worse": fewer copies, or same count and later word
    def __init__(self, count, word):
        self.count, self.word = count, word

    def __lt__(self, other):
        if self.count != other.count:
            return self.count < other.count
        return self.word > other.word


def top_k_words(words, k):
    heap = []                                # STATE: min-heap of the k best words; root = the worst kept
    for word, count in Counter(words).items():
        heapq.heappush(heap, Entry(count, word))         # STEP
        if len(heap) > k:                    # FIX: the worst of k + 1 leaves
            heapq.heappop(heap)
    return [e.word for e in sorted(heap, reverse=True)]  # RETURN: best first


print(k_closest([[3, 3], [5, -1], [-2, 4]], 2))                            # [[-2, 4], [3, 3]]
print(top_k_words(["i", "love", "leetcode", "i", "love", "coding"], 2))    # ['i', 'love']

**Try it**
- Drop the minus in `k_closest` (push `(x * x + y * y, x, y)`): you get the 2 *farthest* points, `[[-2, 4], [5, -1]]`, because the root is now the closest point and it is the one evicted.
- Run `top_k_words(["b", "a", "b", "a"], 1)`: `['a']`. Both words appear twice, and on a tie the later word, `'b'`, is the "worse" one at the root.
- Flip the tie rule in `__lt__` to `self.word < other.word` and rerun that call: `['b']`, the wrong word. The root must be the word you would throw out first.
- Run `k_closest([[1, 1]], 5)`: `[[1, 1]]`. With k above the number of points the cap is never reached, and no special case is needed.

One heap holds one end of the order; the median needs both ends at once. Find Median from Data Stream adds numbers one at a time and asks for the median after each add: after 1 and 2 it is 1.5, after 3 it is 2.0. The median lives at the seam between the smaller half and the larger half, so keep the smaller half in a max-heap and the larger half in a min-heap, and the middle numbers sit at the two roots.

```text
   low (max-heap, stored negated)        high (min-heap)
   1   3   [5]          |                [15]   20
             ^ largest small number       ^ smallest large number
   invariant: every low <= every high, and len(low) is len(high) or len(high) + 1
   median: -low[0] when the count is odd (here 5), else the mean of the two roots
```

`add` keeps both rules in three steps, without a comparison of its own: the number joins the small half, the largest small number crosses the seam, and if the large half is now the bigger one, its smallest number crosses back. `median` then reads only the roots.

In [ ]:
class MedianFinder:
    def __init__(self):
        self.low = []                        # STATE: max-heap (stored negated), the smaller half
        self.high = []                       # STATE: min-heap, the larger half

    def add(self, x):
        heapq.heappush(self.low, -x)                         # STEP: 1. x joins the small half
        heapq.heappush(self.high, -heapq.heappop(self.low))  # FIX: 2. the largest small crosses the seam
        if len(self.high) > len(self.low):                   # FIX: 3. keep len(low) >= len(high)
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def median(self):
        if len(self.low) > len(self.high):   # RETURN: odd count, the extra number sits in low
            return float(-self.low[0])
        return (-self.low[0] + self.high[0]) / 2    # RETURN: even count, the mean of the roots


mf, medians = MedianFinder(), []
for x in [5, 15, 1, 3]:
    mf.add(x)
    medians.append(mf.median())
print(medians)                               # [5.0, 10.0, 5.0, 4.0]

**Try it**
- After each `add`, print `sorted(-v for v in mf.low)` and `sorted(mf.high)`: `[5] []`, `[5] [15]`, `[1, 5] [15]`, `[1, 3] [5, 15]`. Every small number is at most every large one.
- Delete step 3 and rerun: the very first `median()` raises `IndexError`. Step 2 moved the 5 into `high`, and `low` is empty.
- On a fresh `MedianFinder()`, change `/ 2` to `// 2` and add 1, then 2: the median prints `1` instead of `1.5`.
- Predict the medians for `[5, 4, 3, 2, 1]` before running: `[5.0, 4.5, 4.0, 3.5, 3.0]`.

The next variation puts a queue beside the heap, for items that are the best but may not be used yet. Task Scheduler runs one task per tick or idles, with equal tasks at least n ticks apart, and asks for the least total time: `AAABBB` with n = 2 takes 8 ticks, `A B _ A B _ A B`. Always run the ready task with the most copies left, because it is the one that would otherwise force idle ticks at the end.

A task that just ran waits n ticks on a FIFO conveyor belt; tasks leave the belt in the order they got on, so only its front needs checking. The release comes *after* the run: a task whose cooldown ends with this tick may run from the next tick on.

Reorganize String asks for a rearrangement with no two equal neighbours, `aab` to `aba` and `aaab` to `""`, and it is the same machine with a cooldown of one turn. The belt shrinks to a single `held` letter, pushed back only after the next letter has been chosen.

In [ ]:
def least_interval(tasks, n):
    heap = [-c for c in Counter(tasks).values()]   # STATE: -copies left of the tasks ready now (max-heap)
    heapq.heapify(heap)
    cooling = deque()                        # STATE: (tick its cooldown ends, -copies left), oldest first
    time = 0                                 # INIT  (every task with copies left is in exactly one of the two)
    while heap or cooling:
        time += 1                            # one tick
        if heap:
            left = heapq.heappop(heap) + 1   # STEP: run the top (stored negated: +1 means one copy fewer)
            if left:                         # copies remain: it cools down for n ticks
                cooling.append((time + n, left))
        else:
            time = cooling[0][0]             # nothing is ready: skip the idle ticks at once
        if cooling and cooling[0][0] == time:     # FIX: the front's cooldown ends with this tick
            heapq.heappush(heap, cooling.popleft()[1])
    return time                              # RETURN


def reorganize(s):                           # 767: no two equal letters side by side
    heap = [(-c, ch) for ch, c in Counter(s).items()]   # STATE: max-heap of copies left
    heapq.heapify(heap)
    out, held = [], None                     # STATE: held = the letter just placed, sitting out one turn
    while heap:
        c, ch = heapq.heappop(heap)          # STEP: most copies left among the allowed letters
        out.append(ch)
        if held:
            heapq.heappush(heap, held)       # FIX: last turn's letter may be used again
        held = (c + 1, ch) if c + 1 < 0 else None
    return "" if held else "".join(out)      # RETURN: a letter still waiting would have to repeat


print(least_interval(["A", "A", "A", "B", "B", "B"], 2))   # 8   (A B _ A B _ A B)
print(least_interval(["A", "A", "A", "A", "B", "C"], 2))   # 10  (A B C A _ _ A _ _ A)
print(reorganize("aab"), repr(reorganize("aaab")), reorganize("aaabb"))   # aba '' ababa

**Try it**
- Change `(time + n, left)` to `(time + n - 1, left)`: the first call prints 6 instead of 8. Off by one on the ready time silently shrinks the cooldown to n − 1.
- Delete the `else` branch: the answers stay the same, but `least_interval(["A", "A"], 10**6)` now loops a million times; with the jump it returns 1000002 after three loops.
- Check the first call against the counting formula `(most - 1) * (n + 1) + ties` (`most` = the top count, `ties` = how many tasks have it): (3 − 1) × 3 + 2 = 8. The formula needs a `max(len(tasks), ...)` guard; the simulation does not.
- In `reorganize`, push the letter straight back instead of holding it (`heapq.heappush(heap, (c + 1, ch))` right after placing it, when copies remain): `"aab"` comes out as `"aab"`.

The rest of this section is a second pass: Hard problems that reuse the same moves. Skip them until the main path is automatic. The skyline sweeps like [Intervals & Sweep Line](#s14), and IPO, refuelling and Course Schedule III are greedy with regret from [Greedy](#s15): read those first.

The first pattern lets a heap sit beside a sort, so that a greedy choice is made from the right candidates, or can be undone. Sort by the key that *unlocks* options, capital, position or deadline; sweep; push each unlocked option into a heap keyed by its *value*; and pop only when you must choose, or when you must undo a choice.

IPO starts you with capital w and lets you run at most k projects, each needing a minimum capital and paying a profit, to end as rich as possible: k = 2, w = 0, profits `[1, 2, 3]` and capital `[0, 1, 1]` end with 4. Before each choice, push every project the capital now affords, then run the most profitable one.

Minimum Number of Refueling Stops drives a car with some starting fuel toward a target past stations `[position, fuel]`, and asks for the fewest stops, or −1: target 100 with 10 fuel and stations `[[10, 60], [20, 30], [30, 30], [60, 40]]` needs 2. There the fix comes before the step, because you may only take fuel from stations you have actually reached.

Course Schedule III takes courses `[duration, last day]` back to back from day 1 and asks for the most courses that meet their deadlines: `[[100, 200], [200, 1300], [1000, 1250], [2000, 3200]]` allows 3. It takes every course in deadline order and, when a deadline breaks, regrets the longest course taken.

In [ ]:
def max_capital(k, w, profits, capital):                 # IPO (502)
    projects = sorted(zip(capital, profits))             # INIT: unlock order, cheapest first
    heap, i = [], 0                                      # STATE: max-heap (negated) of affordable profits
    for _ in range(k):
        while i < len(projects) and projects[i][0] <= w: # FIX: heap = every affordable, unused project
            heapq.heappush(heap, -projects[i][1])
            i += 1
        if not heap:                                     # nothing affordable, now or ever
            break
        w += -heapq.heappop(heap)                        # STEP: do the most profitable one
    return w                                             # RETURN


def min_refuel_stops(target, fuel, stations):            # (871) fuel = how far we can drive
    passed, stops = [], 0                                # STATE: max-heap (negated) of fuel we drove past
    for pos, gas in stations + [[target, 0]]:            # the target is the last "station"
        while fuel < pos:                                # FIX: stuck before pos, refuel retroactively
            if not passed:
                return -1                                # RETURN: nothing left to take
            fuel += -heapq.heappop(passed)               # the biggest station behind us
            stops += 1                                   # RECORD
        heapq.heappush(passed, -gas)                     # STEP: we reached pos, its fuel is an option
    return stops                                         # RETURN


def schedule_course(courses):                            # (630) courses = [duration, last_day]
    courses = sorted(courses, key=lambda c: c[1])        # INIT: by deadline
    taken, time = [], 0                                  # STATE: max-heap (negated) of taken durations; time = their sum
    for duration, last_day in courses:
        heapq.heappush(taken, -duration)                 # STEP: take it now ...
        time += duration
        if time > last_day:                              # FIX: ... regret, drop the LONGEST course taken
            longest = -heapq.heappop(taken)
            time -= longest
    return len(taken)                                    # RETURN


print(max_capital(2, 0, [1, 2, 3], [0, 1, 1]))                                 # 4
print(min_refuel_stops(100, 10, [[10, 60], [20, 30], [30, 30], [60, 40]]))     # 2
print(schedule_course([[100, 200], [200, 1300], [1000, 1250], [2000, 3200]]))  # 3

**Try it**
- In `min_refuel_stops`, move `heapq.heappush(passed, -gas)` *above* the `while` loop and run `min_refuel_stops(100, 1, [[10, 100]])`: 1 instead of −1. The car refuelled at a station it never reached.
- In `schedule_course`, drop the *shortest* course instead (push `duration` without the minus, pop the root): `schedule_course([[5, 5], [4, 6], [2, 6]])` gives 1 instead of 2. Dropping the longest frees the most time.
- Remove the `if not heap: break` from `max_capital` and run `max_capital(1, 0, [5], [1])`: `IndexError`, since nothing is affordable with 0 capital.
- Print `-passed[0]` each time a stop is taken in the 100-mile example: 60, then 40. The decision is postponed until the car would run dry, and then it is easy.

A heap can't remove an item from its middle; lazy deletion says you don't have to. The Skyline Problem asks for the outline of buildings `[left, right, height]` standing on a shared ground line, as the list of points where the outline's height changes, ending at height 0: `[[2, 9, 10], [3, 7, 15]]` gives `[[2, 10], [3, 15], [7, 10], [9, 0]]`. Sweep the edges left to right with a max-heap of the live buildings.

```text
 15           +-----------+              buildings [2,9,10] [3,7,15] [5,12,12]
 12           |           +--------------+
 10        +--+                          |        sweep the edges left to right;
  0  ------+                             +-----   live = max-heap of (height, right end)
           2  3           7              12       key point = where the top height changes
```

A building that has ended stays in the heap as a dead entry until it reaches the top, because the top is the only thing you ever read. That is why the fix runs before the record reads the top, and why the ground, `(0, math.inf)`, sits in the heap from the start: it never ends, so there is always a top to read.

In [ ]:
def skyline(buildings):
    events = sorted([(l, -h, r) for l, r, h in buildings] +    # INIT: left edge (x, -height, right);
                    [(r, 0, 0) for l, r, h in buildings])      #   a right edge sorts after lefts at x
    live = [(0, math.inf)]                   # STATE: max-heap of (-height, right); the ground never ends
    points = []
    for x, neg_h, right in events:
        while live[0][1] <= x:               # FIX: the tallest has already ended, delete it now
            heapq.heappop(live)
        if neg_h:                            # STEP: a left edge, this building goes live
            heapq.heappush(live, (neg_h, right))
        height = -live[0][0]
        if not points or points[-1][1] != height:   # RECORD: the outline changes here
            points.append([x, height])
    return points                            # RETURN


print(skyline([[2, 9, 10], [3, 7, 15], [5, 12, 12], [15, 20, 10], [19, 24, 8]]))
# [[2, 10], [3, 15], [7, 12], [12, 0], [15, 10], [20, 8], [24, 0]]

**Try it**
- Change `<=` to `<` in the `while` and run `skyline([[0, 2, 3], [2, 4, 1]])`: `[[0, 3], [4, 1]]` instead of `[[0, 3], [2, 1], [4, 0]]`. A building that ends at x does not cover x.
- Remove the `points[-1][1] != height` test (append on every event): the output gains repeats like `[5, 15]` and `[9, 12]`, points where the height did not change.
- Print `x, sorted(live)` right after the `while`: at x = 9 the building `[2, 9, 10]` has ended, yet `(-10, 9)` is still in the heap under the taller `(-12, 12)`. It is harmless there, and it is popped at x = 12 when it surfaces.

The second template, pop the lowest and push its successors, becomes a flood when the frontier is a wall. Trapping Rain Water II asks how much water a 2-D height map holds after rain, when water can only escape over the border: the map in the picture below traps 4. Water in a cell can rise only as high as the lowest wall on its best escape route to the border.

```text
heights            level the water reaches     trapped = level - height
1 4 3 1 3 2        1 4 3 1 3 2                 . . . . . .
3 2 1 3 2 4   ->   3 3 3 3 3 4          ->     . 1 2 0 1 .      total 4
2 3 3 2 3 1        2 3 3 2 3 1                 . . . . . .
```

So start with the border as the wall, always breach the wall at its *lowest* cell, and let the neighbour behind it fill up to that level and join the wall. It is Dijkstra with `max` in place of `+`, the shape you meet again in [Graphs III](09_Graphs.ipynb#s19).

In [ ]:
def trap_rain_water(height):
    m, n = len(height), len(height[0])
    wall = [(height[r][c], r, c) for r in range(m) for c in range(n)
            if r in (0, m - 1) or c in (0, n - 1)]   # STATE + INIT: (level, r, c) of the wall, the border first
    heapq.heapify(wall)
    seen = {(r, c) for _, r, c in wall}      # STATE: cells already in the wall (or retired)
    water = 0
    while wall:
        level, r, c = heapq.heappop(wall)    # STEP: the wall's lowest cell, water escapes here first
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen:
                seen.add((nr, nc))           # mark when pushing, as in BFS
                water += max(0, level - height[nr][nc])                    # RECORD: it fills up to level
                heapq.heappush(wall, (max(level, height[nr][nc]), nr, nc)) # FIX: then it joins the wall
    return water                             # RETURN


print(trap_rain_water([[1, 4, 3, 1, 3, 2], [3, 2, 1, 3, 2, 4], [2, 3, 3, 2, 3, 1]]))   # 4
print(trap_rain_water([[3, 3, 3, 3, 3], [3, 2, 2, 2, 3], [3, 2, 1, 2, 3],
                       [3, 2, 2, 2, 3], [3, 3, 3, 3, 3]]))                          # 10

**Try it**
- Push the neighbour's own height instead of `max(level, height[nr][nc])`: the second grid gives 2 instead of 10. A cell that just filled up rejoins the wall at its low floor height, so its neighbours are filled from that "hole" instead of from the real wall.
- Replace the heap with a FIFO queue (`deque`, `popleft`, `append`): the first grid gives 5 instead of 4. Without "lowest first", a cell gets filled from a wall that is not its real bottleneck.
- Print `level, r, c` for each pop on the first grid: the levels never go down. That monotone order, never decreasing, is what makes each cell's level final when it is first reached.
- Run `trap_rain_water([[5, 1, 5]])`: 0. A grid with fewer than 3 rows or columns has no inside, because every cell is on the border.

The last pattern is the merge template with one more number beside the heap. Smallest Range Covering Elements from K Lists asks for the smallest `[a, b]` that contains at least one number from each of k sorted lists: for `[[4, 10, 15, 24, 26], [0, 9, 12, 20], [5, 18, 22, 30]]` it is `[20, 24]`.

Put one finger on each list. The fingered values cover every list, and they span `[min, max]`. The only way to shrink that range is to raise the min, and only the finger *on* the min can do it: advance it, update the running max, and stop when some list runs out.

In [ ]:
def smallest_range(lists):
    heap = [(lst[0], i, 0) for i, lst in enumerate(lists)]   # STATE + INIT: one finger (value, list, index) per list
    heapq.heapify(heap)
    hi = max(lst[0] for lst in lists)        # STATE: the rightmost finger (a running max)
    best = [-math.inf, math.inf]
    while True:
        lo, i, j = heapq.heappop(heap)       # STEP: the leftmost finger
        if hi - lo < best[1] - best[0]:      # RECORD before moving it (< keeps the earliest start)
            best = [lo, hi]
        if j + 1 == len(lists[i]):           # list i is used up: no later range covers it
            return best                      # RETURN
        nxt = lists[i][j + 1]
        hi = max(hi, nxt)
        heapq.heappush(heap, (nxt, i, j + 1))    # FIX: list i gets its next finger


print(smallest_range([[4, 10, 15, 24, 26], [0, 9, 12, 20], [5, 18, 22, 30]]))   # [20, 24]
print(smallest_range([[1, 2, 3], [1, 2, 3], [1, 2, 3]]))                        # [1, 1]

**Try it**
- Move the "used up" check above the RECORD lines and run `smallest_range([[1], [2], [3]])`: `[-inf, inf]` instead of `[1, 3]`. The last range was never recorded.
- Change `<` to `<=` and run `smallest_range([[1, 3], [2, 4]])`: `[3, 4]` instead of `[1, 2]`. Equal widths must keep the earliest start.
- Delete `hi = max(hi, nxt)`: the first example returns `[20, 5]`, an upside-down "range". `hi` froze at the first fingers' maximum; the right end must follow every new finger.

### Say it in the interview

> "Sorting everything costs O(n log n) and orders items I'm going to throw away. I only need the k best, so I keep a min-heap capped at k: its root is the weakest of my k, and every newcomer only has to beat that root. Anything that leaves was beaten by k numbers still in the heap, so it can't be in the top k.
>
> Each step is O(log k): O(n log k) time, O(k) space, and it works on a stream. For k sorted lists I keep one head per list: every pop is the next item overall, O(N log k)."

While coding, point at the `if len(heap) > k` line and say the invariant: "after this line the heap holds exactly the k largest so far, and `heap[0]` is the k-th largest". In a merge, point at the refill right after the pop: "the list I just took from gets its next item in, so every list always has its head in the heap". And say why the tuple has an index in the middle: "so ties never compare the payload".

Likely follow-ups and your answers:

- *Faster?* → quickselect: O(n) on average, but it needs all the data in memory ([Sorting & Selection](05_Binary_Search_Sorting.ipynb#s23)).
- *k is close to n?* → keep the n − k smallest in a max-heap instead, or heapify everything and pop k times: O(n + k log n).
- *Sorted output?* → pop the k items: O(k log k).
- *Ties or a custom order?* → tuple keys, or a class with `__lt__`.
- *Priorities change?* → push a fresh entry and skip stale ones when they are popped.
- *The data is spread over many machines?* → top k on each machine, then a k-way merge of those lists.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Course Schedule III | `heap/course_schedule_iii.py` | sort by deadline, take every course; past the deadline, drop the longest taken (max-heap) |
| Design Twitter | `heap/design_twitter.py` | each user's tweets are already sorted by time: the feed is a k-way merge, stopped after 10 pops |
| Find Median from Data Stream | `heap/find_median_from_data_stream.py` · `practice/simple/34_find_median_from_data_stream.py` | max-heap of the low half, min-heap of the high half, sizes within 1; the median sits at the roots |
| IPO | `heap/ipo.py` | sort by capital; push newly affordable profits into a max-heap; take the top k times |
| K Closest Points to Origin | `heap/k_closest_points_to_origin.py` · `practice/simple/32_k_closest_points_to_origin.py` | max-heap capped at k on squared distance; the root is the farthest point kept |
| Kth Largest Element in a Stream | `heap/kth_largest_element_in_a_stream.py` | min-heap capped at k; its root is the k-th largest after every add |
| Kth Largest Element in an Array | `heap/kth_largest_element_in_an_array.py` | min-heap capped at k for O(n log k), or quickselect toward index n − k for O(n) average |
| K-th Smallest Prime Fraction | `heap/kth_smallest_prime_fraction.py` | each numerator is a sorted row of fractions: k-way merge, pop k − 1 times |
| Maximum Performance of a Team | `heap/maximum_performance_of_a_team.py` | sort by efficiency, high first (it is the current minimum); min-heap of the k best speeds with a running sum |
| Meeting Rooms III | `heap/meeting_rooms_iii.py` | two heaps: free room ids, busy (end, room); free ended rooms first; a delayed meeting keeps its length |
| Merge k Sorted Lists | `heap/merge_k_sorted_lists.py` · `practice/simple/35_merge_k_sorted_lists.py` | heap of the k heads as (val, i, node); pop the smallest, push its next |
| Minimize Deviation in Array | `heap/minimize_deviation_in_array.py` | double every odd so only halving is left; halve the max (max-heap) and track the min; stop at an odd max |
| Minimum Number of Refueling Stops | `heap/minimum_number_of_refueling_stops.py` | drive past stations into a max-heap; when stuck, retroactively take the biggest |
| Rearrange String k Distance Apart | `heap/rearrange_string_k_distance_apart.py` | max-heap by count + FIFO cooldown queue of length k; an empty heap mid-way means impossible |
| Reorganize String | `heap/reorganize_string.py` | possible iff top count ≤ (n + 1) // 2; place the most frequent letter, holding the last one out a turn |
| Smallest Range Covering Elements from K Lists | `heap/smallest_range_covering_elements_from_k_lists.py` | one finger per list: heap gives the min, a running max the max; advance the min until a list runs out |
| Task Scheduler | `heap/task_scheduler.py` · `practice/simple/33_task_scheduler.py` | run the ready task with most copies left (max-heap); park it in a FIFO cooldown queue |
| The Skyline Problem | `heap/the_skyline_problem.py` | sweep the edges; max-heap of live (height, right) with lazy deletion; emit when the top height changes |
| Top K Frequent Words | `heap/top_k_frequent_words.py` | size-k heap whose root is the worst word (`__lt__`: fewer copies, or later word); or heapify (−count, word), pop k |
| Trapping Rain Water II | `heap/trapping_rain_water_ii.py` | the border is a wall; breach its lowest cell (min-heap); neighbours fill to that level and join the wall |

### Self-check

1. You want the 10 largest numbers of a stream of a billion. Which heap, which size, and what does its root mean?
<details><summary>Answer</summary>A min-heap capped at 10. Its root is the smallest of the 10 largest seen so far (the 10th largest), so a new number stays only if it beats the root, and then the root is popped. O(log 10) per number and O(10) memory.</details>

2. Why does the keep-k template push first and evict second?
<details><summary>Answer</summary>After the push the heap holds k + 1 candidates, and popping removes the weakest of all of them, which may be the newcomer itself. Evicting first removes a kept item before the newcomer has been compared with anything: <code>k_largest([5, 1], 1)</code> would keep 1.</details>

3. For the k closest points, why cap a max-heap at k instead of heapifying all n into a min-heap?
<details><summary>Answer</summary>The capped max-heap's root is the farthest point kept, so each newcomer is compared once, with that root, and memory stays O(k): O(n log k) time, and it works on a stream. Heapifying all n takes O(n) memory and O(n + k log n) time, and it needs every point up front.</details>

4. In the median finder, why does every number go into `low` first and then move `low`'s maximum to `high`?
<details><summary>Answer</summary>That two-step dance keeps the ordering rule without any comparisons of your own: whatever crosses the seam is the largest of the small half (including the new number), so every number in <code>low</code> stays at most every number in <code>high</code>. Step 3 then fixes only the sizes.</details>

<a id="s14"></a>

## Intervals & Sweep Line

> Lay the intervals on a number line and sweep a vertical line from left to right. Sort once, and every decision only needs what the line is touching right now: the block being merged, the room that frees up first, or a running count of open intervals.

The heap handed you the cheapest item next. Intervals add a second habit, sorting once: after one sort by start, a single left-to-right sweep answers most interval questions, and a heap of end times returns when you must know which meeting finishes first.

**Reach for it when** the input is pairs `[start, end]` (meetings, bookings, ranges, buildings, rectangles) and the question is about overlap: merge them, insert one, intersect two lists of them, keep the most / remove the fewest so that none overlap, find how many are open at once (rooms), find the free gaps, or answer "which intervals contain point q?".

**In this repo:** `intervals/` (8 problems) · bank: `practice/simple/48_merge_intervals.py`, `practice/simple/49_meeting_rooms_ii.py` · basics: `practice/simple/basics/sorting/06_python_sort_keys_and_stability.py` (sort keys such as `(end, -start)`), `practice/simple/basics/heaps/` (the min-heap behind meeting rooms) · sweeps that need a heap live in [Heaps](#s13): The Skyline Problem and Meeting Rooms III; covering a range with the fewest taps and "two points per interval" live in [Greedy](#s15); [Two Pointers](03_Two_Pointers_Sliding_Window_Strings.ipynb#s05) walks two sorted lists the way Interval List Intersections does below.

### The picture

```text
0   2   4   6   8   10  12  14  16  18
  [===]                                     [1,3]
    [=======]                               [2,6]    2 <= 3: same block, its end stretches to 6
                [===]                       [8,10]   8 > 6: block [1,6] is final, open a new one
                  [=====]                   [9,12]   9 <= 10: stretch to 12
                              [=====]       [15,18]  15 > 12: block [8,12] is final
  [=========]   [=======]     [=====]       merged:  [1,6] [8,12] [15,18]
```

Compare every pair and you pay O(n²); merge until nothing changes and it is O(n³). After sorting by start, an interval can only overlap the block you are building *right now*: every later interval starts even later, so once one starts past the block's end, that block is final and never looked at again. One sort and one sweep: O(n log n).

### Overlap in one line

```text
two intervals overlap  <=>  each one starts before the other one ends

  a [=======)              a [=======)                a [=====)
  b     [=======)          b         [=======)        b           [=====)
    overlap                  touching: a.end == b.start   apart

half-open [s, e)   a.start <  b.end and b.start <  a.end     meetings, bookings
closed    [s, e]   a.start <= b.end and b.start <= a.end     merge intervals, "touching overlaps"
```

It is easier to say when two intervals do *not* overlap: one ends before the other starts. Negate that and you get the one-liner above, which handles partial, nested and identical intervals with no case analysis. When they do overlap, the common part is `[max(a.start, b.start), min(a.end, b.end)]`.

The only real question is what happens at a shared endpoint, and the problem statement decides it. Take `[1, 2]` and `[2, 3]`. Merge Intervals (56) and Insert Interval (57), which merge whatever overlaps, treat ends as closed: the two form one block `[1, 3]`, and the test is `start <= block_end`. Interval List Intersections (986), the common parts of two lists, is closed too: the two share the point 2, and a pair meets when `max(starts) <= min(ends)`.

Rooms and bookings are half-open, `[s, e)`. In Meeting Rooms II (253), the fewest rooms that hold a set of meetings, and in My Calendar I (729) and III (732), which take bookings one at a time, a meeting that ends at 2 leaves its room free for one that starts at 2, so a room is free when `end <= start`.

Two more problems read touching the same way. Non-overlapping Intervals (435), which removes the fewest intervals so that none overlap, keeps `[1, 2]` and `[2, 3]` together: an interval stays when `start >= kept_end`. Employee Free Time (759), the gaps when everyone is free, finds no gap between busy blocks that touch, so a gap opens only when `start > busy_until`.

### From idea to code

*Sort by start, then sweep: to merge, keep one "current" block that each interval either stretches or closes; to count, keep the end times of what is still running, free what has ended, then add the newcomer.*

For merging, as in Merge Intervals and Insert Interval, the **State** is the block being built, `merged[-1]`, whose **Definition** is the union of everything so far that chains into it; every block before it is finished. The **Invariant** is that `merged` is sorted, disjoint, and covers exactly the intervals seen so far. A **Step** stretches the block to `max(merged[-1][1], end)`, the `max` because the newcomer may be nested inside, or opens a new block when the newcomer starts past the block's end.

There is no **Fix**, because after the sort only `merged[-1]` can be touched. The **Record** happens when a block becomes final: the moment an interval starts after its end, or when the input ends. The **Init** is the sort by start, `sorted(intervals)`, which orders lists by start and then by end, and an empty `merged`; the **Return** is `merged`, `[]` for empty input.

For counting how many run at once, as in Meeting Rooms II, the state is `ends`, a min-heap of the end times of the meetings still running at the current start, so `ends[0]` is the room that frees up first. The invariant, after the fix, is that every end in the heap is later than `start`.

The fix frees every room whose meeting is over, `while ends and ends[0] <= start: heappop(ends)`. The step pushes the newcomer's end, because it takes a room, and the record is `rooms = max(rooms, len(ends))` right after the push. Init sorts by start with an empty heap; the return is `rooms`.

The record reads how many meetings run at this instant, so it must come after both other lines. Move the freeing loop below the count and the back-to-back meetings `[1, 5]`, `[5, 9]` need 2 rooms; move the count above the push and `[[1, 5], [2, 6], [3, 7]]` needs only 2 rooms instead of 3. Freeing before the push also reads naturally: the newcomer may take a room whose meeting ended exactly at its start.

The cell below holds both templates. Merge Intervals merges every overlapping pair, touching included: `[[1, 3], [2, 6], [8, 10], [15, 18]]` becomes `[[1, 6], [8, 10], [15, 18]]`. Meeting Rooms II asks for the fewest rooms so that no two meetings share a room at the same time, where a meeting may start exactly when another ends: `[[0, 30], [5, 10], [15, 20]]` needs 2. The `overlaps` helper is the one-liner for half-open ends.

In [ ]:
def merge(intervals):
    intervals = sorted(intervals)                     # INIT: by start (ties by end)
    merged = []                                       # STATE: finished blocks + the current one, merged[-1]
    for start, end in intervals:
        if merged and start <= merged[-1][1]:         # starts inside the current block:
            merged[-1][1] = max(merged[-1][1], end)   # STEP: stretch it (max: it may be nested)
        else:
            merged.append([start, end])               # STEP + RECORD: the block before is final; open a new one
    return merged                                     # RETURN


def overlaps(a, b):                                   # half-open [s, e): touching is NOT overlapping
    return a[0] < b[1] and b[0] < a[1]


def min_rooms_heap(meetings):
    meetings = sorted(meetings)              # INIT: by start
    ends, rooms = [], 0                      # STATE: min-heap of end times of the meetings running now
    for start, end in meetings:
        while ends and ends[0] <= start:     # FIX: free every room whose meeting is over (half-open: <=)
            heapq.heappop(ends)
        heapq.heappush(ends, end)            # STEP: this meeting takes a room
        rooms = max(rooms, len(ends))        # RECORD: rooms in use right now
    return rooms                             # RETURN


print(merge([[1, 3], [2, 6], [8, 10], [15, 18]]))     # [[1, 6], [8, 10], [15, 18]]
print(merge([[1, 4], [4, 5]]), overlaps([1, 4], [4, 5]), overlaps([1, 9], [3, 4]))   # [[1, 5]] False True
print(min_rooms_heap([[0, 30], [5, 10], [15, 20]]), min_rooms_heap([[1, 5], [5, 9]]))  # 2 1

**Try it**
- Change `max(merged[-1][1], end)` to plain `end` and run `merge([[1, 10], [2, 3], [4, 5]])`: `[[1, 3], [4, 5]]`. The nested `[2, 3]` shrank the block.
- Change `<=` to `<` in `merge`: `merge([[1, 4], [4, 5]])` now gives `[[1, 4], [4, 5]]`. Closed or half-open is a decision you read from the problem, then write once.
- Delete the INIT line, so that the loop sees the input order, and run `merge([[8, 10], [1, 3], [2, 6]])`: `[[8, 10]]`. The test only looks at the block's end, so `[1, 3]` and `[2, 6]` were swallowed by a block they never touch.
- In `min_rooms_heap`, move the `while` loop below the RECORD line: `min_rooms_heap([[1, 5], [5, 9]])` needs 2 rooms. The room was freed one step too late.

The peak is the answer, and the reason is worth saying out loud. At the busiest instant that many meetings run at once, and each needs its own room, so fewer rooms is impossible. That many is also enough: hand out rooms in start order, each to a room whose meeting has ended. One is always free, because otherwise one more meeting than the peak would be running at that moment.

### Watch it work

Fed in a shuffled order, the sweep still sees the intervals sorted. The trace prints, for each interval, whether it stretched the open block or closed it, and the blocks built so far.

In [ ]:
def trace_merge(intervals):
    merged = []
    for start, end in sorted(intervals):
        if merged and start <= merged[-1][1]:
            old = merged[-1][1]
            merged[-1][1] = max(old, end)
            note = f"{start} <= {old}: stretch to {merged[-1][1]}"
        else:
            note = f"{start} > {merged[-1][1]}: {merged[-1]} is final" if merged else "open the first block"
            merged.append([start, end])
        print(f"[{start:>2},{end:>3}]  {note:<26} merged={merged}")


trace_merge([[8, 10], [1, 3], [15, 18], [2, 6], [9, 12]])

**Try it**
- Run `trace_merge([[1, 10], [2, 3], [4, 5]])`: the block stays `[1, 10]` while the small intervals are swallowed. That is the `max` at work.
- Run `trace_merge([[1, 2], [2, 3], [3, 4]])`: each touches the next, so the chain becomes one block `[1, 4]`.
- Run `trace_merge([[1, 2], [3, 4], [2, 3]])`: one block `[1, 4]`. Without the sort the loop gives `[[1, 2], [3, 4]]`: a late interval can bridge two blocks that were already closed.

### Where it goes wrong

Most interval bugs are one wrong comparison, so each trap below names the comparison and the input that exposes it.

1. **Sorting by the wrong key, or not at all.** Merging and rooms: sort by start. "Keep the most / remove the fewest": sort by **end**. With `[[1, 100], [2, 3], [4, 5]]`, keeping greedily in start order keeps `[1, 100]` and removes 2; end order removes only 1.
2. **`end` instead of `max(...)`** when stretching: a nested interval shrinks the block (`[[1, 10], [2, 3], [4, 5]]` → `[[1, 3], [4, 5]]`).
3. **Comparing with the previous interval instead of the block.** After the nested `[2, 3]`, the previous end (3) is below the block's end (10), so `[[1, 10], [2, 3], [4, 5]]` becomes `[[1, 10], [4, 5]]`. Always compare with `merged[-1]`.
4. **`<` vs `<=` at a shared endpoint.** Closed ends (merge): `[1, 4]` and `[4, 5]` overlap. Half-open (meetings): a meeting ending at 10 frees its room for one starting at 10.
5. **Editing the caller's lists.** `merged.append(iv)` followed by `merged[-1][1] = ...` changes the input in place: after `merge([[1, 3], [2, 4]])` the caller's `[1, 3]` reads `[1, 4]`. Append a fresh `[start, end]`.
6. **Offline answers in the wrong order.** If you sort the queries to sweep them, store each answer under its original query, then rebuild the original order. Returned in sorted order, the queries `[2, 19, 5, 22]` of `min_interval` below get `[2, 4, -1, 6]` instead of `[2, -1, 4, 6]`.
7. **The +1/−1 order at equal times.** For half-open intervals the −1 must come before the +1 at the same time, or back-to-back meetings count as overlapping: `[[1, 5], [5, 9]]` needs 2 rooms instead of 1. Sorting `(time, delta)` tuples does it for free: −1 sorts first.
8. **Adding a strip with the new set.** In a 2-D sweep, as in Rectangle Area II, the area of a union of rectangles, add the strip to the left of x with the *old* active set, then apply the event at x. The other order loses area: on the three rectangles of the example at the end of the section, the last strip `[2, 3]` is measured after `[1, 0, 3, 1]` has switched off, and the area comes out 5 instead of 6.

### Edge cases to say out loud

Empty list · one interval · touching ends (closed vs half-open) · nested intervals · identical intervals · unsorted input · zero-length intervals `[5, 5]` · negative coordinates · inserting before all, after all, or around all. The cell checks the merge and rooms templates against them, and the insert cases come with Insert Interval below.

In [ ]:
assert merge([]) == []
assert merge([[5, 5]]) == [[5, 5]]                          # zero length is still an interval
assert merge([[1, 10], [2, 3]]) == [[1, 10]]                # nested: max keeps the 10
assert merge([[8, 9], [1, 3], [2, 4]]) == [[1, 4], [8, 9]]  # unsorted input
assert merge([[1, 2], [1, 2]]) == [[1, 2]]                  # identical
assert merge([[-5, -1], [-2, 0]]) == [[-5, 0]]
assert not overlaps([1, 2], [2, 3]) and overlaps([1, 3], [2, 2.5])
assert min_rooms_heap([]) == 0 and min_rooms_heap([[1, 2]]) == 1
ivs = [[1, 3], [2, 4]]
merge(ivs)
assert ivs == [[1, 3], [2, 4]]                              # the caller's lists are untouched
print("edge cases pass")

**Try it**
- Predict `merge([[1, 4], [0, 0]])` before running it: `[[0, 0], [1, 4]]` (they don't touch).
- Rewrite the loop as `for iv in intervals:` with `merged.append(iv)` and `iv[0]`, `iv[1]` inside: the last assert fails, because the caller's `[1, 3]` became `[1, 4]`. `sorted` copied the outer list, not the inner ones.
- What should `min_rooms_heap([[1, 3], [1, 3], [1, 3]])` return? Write the assert first (3).

### Variations

Every variation below changes one of three things: the sort key, what the sweep carries, or whether the input arrives all at once.

| Variation | What changes from the template | Problems |
|---|---|---|
| **Merge** | the template | Merge Intervals (56) |
| **Insert into a sorted, disjoint list** | no sort: copy what ends before, swallow what overlaps, copy the rest | Insert Interval (57) |
| **Keep the most / remove the fewest** | sort by END; keep an interval if it starts at or after the last kept end | Non-overlapping Intervals (435); Minimum Number of Arrows to Burst Balloons (452): the fewest arrows that burst every balloon |
| **How many at once** | the rooms template, or +1/−1 events with ends first at equal times; Car Pooling compares the peak with a capacity | Meeting Rooms II (253); Car Pooling (1094): whether one car of a given capacity can carry every trip |
| **Intersect two sorted lists** | two pointers; the common part is `[max starts, min ends]`; advance whichever ends first | Interval List Intersections (986) |
| **Free gaps** | merge everyone's busy time; the holes between blocks are free | Employee Free Time (759) |
| **Bookings one at a time** | keep accepted bookings sorted; check only the two neighbours with `bisect` | My Calendar I (729) |
| **Covered intervals** | sort by `(start, -end)`; an interval is covered iff its end ≤ the largest end so far | Remove Covered Intervals (1288): how many intervals no other interval contains |
| **Cover a range with the fewest intervals** | jump-game reach, in [Greedy](#s15) | Minimum Number of Taps to Open to Water a Garden (1326): the fewest taps that water a garden `[0, n]` |
| **At least two points in every interval** | sort by end, place points at the right end, in [Greedy](#s15) | Set Intersection Size At Least Two (757): the fewest points that hit every interval twice |
| **Rooms with ids, delayed meetings** | two heaps, free ids and busy ends, in [Heaps](#s13) | Meeting Rooms III (2402): the room that hosts the most meetings |
| *Second pass:* **point queries** | sort the queries offline; push intervals that have started; lazily pop those that ended | Minimum Interval to Include Each Query (1851): the smallest interval holding each query point |
| *Second pass:* **running bookings** | difference map: +1 at start, −1 at end; sweep the sorted keys | My Calendar II (731): no triple bookings; My Calendar III (732) |
| *Second pass:* **area of a union** | sweep x; each strip adds width × union length of the active y-intervals | Rectangle Area II (850) |

The first variation drops the sort. Insert Interval is given a sorted, disjoint list and one new interval, and asks for the list with the new interval merged in: `[[1, 3], [6, 9]]` with `[2, 5]` becomes `[[1, 5], [6, 9]]`. The intervals that touch the new one form a single run.

```text
[1,2]   [3,5]   [6,7]   [8,10]   [12,16]          new = [4,8]
copy    swallow swallow swallow  copy             ->  [1,2] [3,10] [12,16]
```

So the code has three phases and no sort: copy every interval that ends before the new one starts, swallow every interval that starts at or before its end, touching included, and copy the rest. Each interval is looked at once.

In [ ]:
def insert(intervals, new):
    out, i, n = [], 0, len(intervals)        # STATE: out = the answer so far; i = next interval to look at
    start, end = new                         # STATE: the new interval, growing as it swallows others
    while i < n and intervals[i][1] < start:     # 1. ends before new starts: copy it untouched
        out.append(intervals[i])
        i += 1
    while i < n and intervals[i][0] <= end:      # 2. overlaps (touching counts):
        start = min(start, intervals[i][0])      # STEP: swallow it
        end = max(end, intervals[i][1])
        i += 1
    out.append([start, end])                     # RECORD: the grown new interval
    return out + intervals[i:]                   # RETURN: 3. the rest starts after new ends


print(insert([[1, 3], [6, 9]], [2, 5]))                              # [[1, 5], [6, 9]]
print(insert([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]))   # [[1, 2], [3, 10], [12, 16]]

**Try it**
- Insert at the very end, `insert([[1, 2]], [5, 6])`, at the very front, `insert([[3, 4]], [1, 2])`, and around everything, `insert([[3, 4], [5, 6]], [1, 9])`: phase 2 is empty in the first two calls and swallows both intervals in the third, and the `append` between the loops lands in the right place every time.
- Change phase 1's `<` to `<=` and run `insert([[1, 2]], [2, 3])`: `[[1, 2], [2, 3]]` instead of `[[1, 3]]`. The touching interval was copied instead of swallowed.
- Run `insert([], [4, 8])`: `[[4, 8]]`, with both loops skipped. This is O(n) while `merge` is O(n log n); point at the line that is missing.

The second variation changes the sort key. Non-overlapping Intervals asks for the fewest intervals to remove so that the rest are pairwise disjoint, touching allowed: `[[1, 2], [2, 3], [3, 4], [1, 3]]` removes 1. To fit the most intervals, always keep the one that *ends* first, because it leaves the most room for everything after it.

The proof is an exchange argument, the greedy proof that [Greedy](#s15) spells out: any best answer can swap its first interval for the earliest-ending one without creating a clash, so the greedy never loses. In code, sort by end once, keep an interval when it starts at or after the last kept end, and count every other interval as a removal.

In [ ]:
def erase_overlap_intervals(intervals):
    intervals = sorted(intervals, key=lambda iv: iv[1])   # INIT: earliest END first
    kept_end, removed = -math.inf, 0         # STATE: kept_end = end of the last kept interval
    for start, end in intervals:
        if start >= kept_end:                # fits after the last kept one (touching is fine)
            kept_end = end                   # STEP: keep it
        else:
            removed += 1                     # RECORD: it clashes with a kept one that ends no later
    return removed                           # RETURN


print(erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]))   # 1
print(erase_overlap_intervals([[1, 2], [1, 2], [1, 2]]))           # 2
print(erase_overlap_intervals([[1, 100], [2, 3], [4, 5]]))         # 1

**Try it**
- Sort by start instead (`key=lambda iv: iv[0]`) and rerun the last call: 2 instead of 1. `[1, 100]` is kept first and blocks everything.
- Sort by length (`key=lambda iv: iv[1] - iv[0]`) and run `erase_overlap_intervals([[1, 5], [4, 6], [5, 10]])`: 2 instead of 1. The short `[4, 6]` clashes with both of the others.
- Change `>=` to `>`: `erase_overlap_intervals([[1, 2], [2, 3]])` now removes 1 instead of 0, as if touching intervals overlapped.
- Adapt the loop to Minimum Number of Arrows to Burst Balloons (452), where each balloon is a closed interval and one arrow bursts every balloon it passes through: sort by end and shoot a new arrow only when `start > arrow`. `[[10, 16], [2, 8], [1, 6], [7, 12]]` needs 2 arrows, and so does `[[1, 2], [2, 3], [3, 4], [4, 5]]`.

The rooms count has a second form, with no heap at all. The room count is the largest number of meetings running at the same instant, so turn each meeting into two events, +1 at its start and −1 at its end, and keep a running total: `[[0, 30], [5, 10], [15, 20]]` peaks at 2.

```text
meetings [0,30) [5,10) [15,20)
time      0    5   10   15   20   30
event    +1   +1   -1   +1   -1   -1
open      1    2    1    2    1    0          peak = 2 rooms
```

Sort the events as `(time, delta)` tuples, so that at equal times the −1 comes first and a room freed at 5 is free for a meeting that starts at 5. The running total after each event is the number of meetings open, and its peak is the answer.

In [ ]:
def min_rooms_sweep(meetings):
    events = sorted([(s, 1) for s, e in meetings] + [(e, -1) for s, e in meetings])   # INIT: (t, -1) before (t, 1)
    open_now = rooms = 0                     # STATE: open_now = meetings running after this event
    for _, delta in events:
        open_now += delta                    # STEP
        rooms = max(rooms, open_now)         # RECORD
    return rooms                             # RETURN


print(min_rooms_sweep([[0, 30], [5, 10], [15, 20]]), min_rooms_sweep([[1, 5], [5, 9]]))   # 2 1

**Try it**
- Sort on time only, `sorted(..., key=lambda ev: ev[0])`: `[[1, 5], [5, 9]]` needs 2 rooms. Python's sort is stable, so at time 5 the start (listed first) is now processed before the end.
- Print `events` for the first call and run the totals by hand: 1, 2, 1, 2, 1, 0.
- Compare with the heap on random meetings (`start < end`): the two always agree, because both measure the largest number of meetings open at once.

The two forms answer different follow-ups. The heap knows *which* room frees up next, which Meeting Rooms III needs when it hands each meeting the lowest free room; the sweep is shorter and also works when bookings arrive one at a time, as in My Calendar III below. Car Pooling asks whether a car with a given capacity can serve trips `[passengers, from, to]`, and it is this sweep with the peak compared against the capacity.

Meeting Rooms II promises `start < end`, and the promise matters: with zero-length meetings the two forms disagree, `[[5, 5], [5, 5]]` being 1 for the heap and 0 for the sweep.

Two sorted lists need no sweep line at all, only two fingers. Interval List Intersections asks for the common parts of two sorted, disjoint lists of closed intervals: `[[0, 2], [5, 10]]` and `[[1, 5], [8, 12]]` share `[[1, 2], [5, 5], [8, 10]]`. The current pair meets on `[max of the starts, min of the ends]` when that is non-empty. Then drop the interval that ends first: it can't meet anything later in the other list, because everything there starts even later.

In [ ]:
def interval_intersection(A, B):             # 986: both sorted and disjoint, closed ends
    i = j = 0                                # STATE: the current interval of each list
    out = []
    while i < len(A) and j < len(B):
        lo = max(A[i][0], B[j][0])           # the common part starts at the later start
        hi = min(A[i][1], B[j][1])           # ... and ends at the earlier end
        if lo <= hi:                         # RECORD: they meet (touching counts)
            out.append([lo, hi])
        if A[i][1] < B[j][1]:                # STEP: the one that ends first can't meet anything later
            i += 1
        else:
            j += 1
    return out                               # RETURN


print(interval_intersection([[0, 2], [5, 10], [13, 23], [24, 25]], [[1, 5], [8, 12], [15, 24], [25, 26]]))
# [[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]

**Try it**
- Change `lo <= hi` to `lo < hi`: `[[1, 2], [8, 10], [15, 23]]`. The single-point intersections `[5, 5]`, `[24, 24]`, `[25, 25]` are lost.
- Advance the pointer that ends *later* (swap `i += 1` and `j += 1`): only `[[1, 2]]` is found. You threw away the interval that could still meet the next one.
- Print `i, j` at the top of the loop: each step moves exactly one pointer, so the loop runs at most `len(A) + len(B)` times.

The merge template also answers questions about what is *not* covered. Employee Free Time gives each employee's sorted busy intervals and asks for the finite gaps when *everyone* is free: `[[[1, 2], [5, 6]], [[1, 3]], [[4, 10]]]` gives `[[3, 4]]`. Free time for everyone is what is left after the union of everyone's busy time, so merge all the busy intervals; the holes between consecutive blocks are the answer.

In [ ]:
def employee_free_time(schedules):
    busy = merge([iv for person in schedules for iv in person])   # STATE: everyone's busy time, merged
    return [[a[1], b[0]] for a, b in zip(busy, busy[1:])]         # RETURN: the holes between blocks


print(employee_free_time([[[1, 2], [5, 6]], [[1, 3]], [[4, 10]]]))          # [[3, 4]]
print(employee_free_time([[[1, 3], [6, 7]], [[2, 4]], [[2, 5], [9, 12]]]))  # [[5, 6], [7, 9]]

**Try it**
- Add a person who works `[[3, 4]]` to the first call: the answer becomes `[]`. Touching blocks merge (`<=`), so no zero-length gap can appear.
- Print `busy` for the second call: `[[1, 5], [6, 7], [9, 12]]`, and read the gaps straight off it.
- Each employee's list is already sorted, so rewrite it as a heap k-way merge, one head per employee like `merge_sorted` in [Heaps](#s13): pop the earliest start, record a gap if it starts after the latest end so far, push that employee's next interval. It prints the same answers in O(N log k) instead of O(N log N). On LeetCode the intervals arrive as `Interval` objects, so read `.start` and `.end` there.

When bookings arrive one at a time, the sweep gives way to a sorted list. My Calendar I is a class-shaped question, like those in [Design Problems](11_Design.ipynb#s24): `book(start, end)` accepts a half-open booking and returns `True` only if it overlaps no accepted booking, so `(10, 20)` is accepted, `(15, 25)` refused and `(20, 30)` accepted. Keep the accepted bookings sorted; then only the two neighbours of the new booking can overlap it, the one just before, if it ends after `start`, and the one just after, if it starts before `end`.

In [ ]:
class MyCalendar:                                # 729
    def __init__(self):
        self.starts, self.ends = [], []          # STATE: accepted bookings, sorted and disjoint

    def book(self, start, end):                  # half-open [start, end)
        i = bisect.bisect_right(self.starts, start)          # bookings 0..i-1 start at or before `start`
        if (i > 0 and self.ends[i - 1] > start) or (i < len(self.starts) and self.starts[i] < end):
            return False                         # RETURN: it overlaps a neighbour
        self.starts.insert(i, start)             # STEP: keep both lists sorted
        self.ends.insert(i, end)
        return True                              # RETURN


cal1 = MyCalendar()
print([cal1.book(s, e) for s, e in [(10, 20), (15, 25), (20, 30)]])   # [True, False, True]

**Try it**
- Then book `(5, 10)`: `True`. It ends exactly where `(10, 20)` starts, and the ends are half-open.
- Change `self.ends[i - 1] > start` to `>=`: the third booking `(20, 30)` is now rejected (`[True, False, False]`). Touching bookings clash.
- Check only the booking before (drop the second condition): on a fresh calendar holding `(20, 30)`, `book(15, 25)` is accepted, although they overlap.

The rest of this section is a second pass: Hard problems that reuse the same moves. Skip them until the main path is automatic. They are offline queries, a difference map and a sweep over rectangles, built from the sweep and the heap of the core.

Minimum Interval to Include Each Query gives intervals `[left, right]` and query points, and asks for the size, `right − left + 1`, of the smallest interval containing each query, or −1: intervals `[[1, 4], [2, 4], [3, 6], [4, 4]]` with queries `[2, 3, 4, 5]` give `[3, 3, 1, 4]`. Answer the queries *offline*, which means all of them are known up front and may be answered in whatever order is convenient, here increasing.

```text
intervals sorted by left: [1,4] [2,4] [3,6] [4,4]          (sizes 4, 3, 4, 1)
q = 2: push [1,4] [2,4]                          top [2,4]  -> 3
q = 3: push [3,6]                                top [2,4]  -> 3
q = 4: push [4,4]                                top [4,4]  -> 1
q = 5: pop [4,4], [2,4], [1,4] (they ended)      top [3,6]  -> 4
```

As the query point moves right, intervals that have *started* join a min-heap keyed by size. Intervals that have *ended* are dead for this and every later query, so they are popped lazily, only when they reach the top. A dict keeps each query's answer, which puts the answers back in the original order at the end.

In [ ]:
def min_interval(intervals, queries):
    intervals = sorted(intervals)                # INIT: by left end
    heap, i, best = [], 0, {}                    # STATE: heap = (size, right) of intervals that have started
    for q in sorted(set(queries)):               # offline: queries in increasing order
        while i < len(intervals) and intervals[i][0] <= q:     # every interval that has started by q ...
            left, right = intervals[i]
            heapq.heappush(heap, (right - left + 1, right))    # STEP: ... joins the candidates
            i += 1
        while heap and heap[0][1] < q:           # FIX: lazy deletion, the top ended before q
            heapq.heappop(heap)
        best[q] = heap[0][0] if heap else -1     # RECORD
    return [best[q] for q in queries]            # RETURN: back in the original order


print(min_interval([[1, 4], [2, 4], [3, 6], [4, 4]], [2, 3, 4, 5]))      # [3, 3, 1, 4]
print(min_interval([[2, 3], [2, 5], [1, 8], [20, 25]], [2, 19, 5, 22]))  # [2, -1, 4, 6]

**Try it**
- Change `heap[0][1] < q` to `<=`: the first call gives `[3, 3, 4, 4]`. A query equal to an interval's right end is inside it, so `[4, 4]` must answer q = 4.
- Return `[best[q] for q in sorted(queries)]` instead: the second call gives `[2, 4, -1, 6]`, right numbers in the wrong order.
- Print `heap` after the expiry loop for each query of the second call: at q = 19 everything has ended, the heap is empty, and the answer is −1.

The +1/−1 sweep also works online, one booking at a time. My Calendar III accepts every booking and, after each one, reports the largest number of bookings that overlap at some instant: after `(10, 20)`, `(50, 60)` and `(10, 40)` the answer is 2, and `(5, 15)` makes it 3. The number of events open at time t is the starts at or before t minus the ends at or before t.

So store only the changes, +1 at each start and −1 at each end; sweeping the keys in sorted order and summing rebuilds the count everywhere, and its peak is the answer. That is a difference array from [Prefix Sums](02_Arrays_Hashing_Prefix_Sums.ipynb#s04), a list of changes rather than of values, here on a sparse, unbounded timeline.

Each `book` below sorts the keys, O(n log n). A key list kept sorted with `bisect.insort` makes it O(n), and a lazy segment tree over the coordinate range C makes it O(log C).

In [ ]:
class MyCalendarThree:
    def __init__(self):
        self.delta = defaultdict(int)            # STATE: time -> change in the number of open events

    def book(self, start, end):                  # half-open [start, end)
        self.delta[start] += 1                   # STEP: one more event from start ...
        self.delta[end] -= 1                     # ... until end
        open_now = best = 0
        for t in sorted(self.delta):             # sweep the boundaries left to right
            open_now += self.delta[t]
            best = max(best, open_now)           # RECORD
        return best                              # RETURN


cal = MyCalendarThree()
print([cal.book(s, e) for s, e in [(10, 20), (50, 60), (10, 40), (5, 15), (5, 10), (25, 55)]])
# [1, 1, 2, 3, 3, 3]

**Try it**
- On a fresh calendar, book `(1, 5)` and then `(5, 9)`: both return 1. At time 5 the −1 and the +1 land on the same key and cancel, which is the half-open rule.
- Print `sorted(cal.delta.items())` after the six bookings and add the values up by hand: the running sum goes 2, 3, 2, 1, 2, 1, 2, 1, 0, so the peak is 3.
- Turn the class into My Calendar II (731), which rejects a booking that would make a triple booking: add the two deltas, sweep, and if the peak reaches 3, take them back and return `False`. The six bookings above now give `[True, True, True, False, True, True]`.

The last pattern turns the sweep sideways and runs the merge inside it. Rectangle Area II asks for the area covered by the union of axis-aligned rectangles `[x1, y1, x2, y2]`, modulo 10⁹ + 7, with overlaps counted once: `[[0, 0, 2, 2], [1, 0, 2, 3], [1, 0, 3, 1]]` covers 6. Python's integers never overflow, so a single modulo at the end is enough.

Area is a sum of thin strips. Sweep a vertical line through the x-edges: between two consecutive edges the set of rectangles the line cuts does not change, so that strip adds its width times the union length of their y-intervals, the merge template in one dimension. In the example the strips `[0, 1]`, `[1, 2]` and `[2, 3]` cover heights 2, 3 and 1, so the area is 6. Measure each strip with the old set, before the event at its right edge switches a rectangle on or off; that is trap 8.

### Say it in the interview

> "Comparing every pair is O(n²). If I sort by start, an interval can only overlap the block I'm currently building, because everything after it starts even later. So I sort once and sweep once, keeping `merged[-1]` as the open block: O(n log n) time, O(n) for the output.
>
> For 'how many rooms', I keep a min-heap of the end times of running meetings: free the ended ones, add the newcomer, and the largest heap size is the answer. Fewer rooms is impossible, because at that moment that many meetings run at once; and it is enough, because a newcomer always finds a room whose meeting has ended."

Say which ends are closed *before* you write the comparison ("touching intervals merge, so `<=`"), and point at the `max` when you stretch: "max, because the next interval may be nested inside the block".

Likely follow-ups and your answers:

- *Do touching intervals overlap?* → ask. Closed ends mean `<=`, half-open ends mean `<`.
- *The list is already sorted, insert one interval?* → three phases, O(n), no sort.
- *Which room does each meeting get?* → a heap of `(end, room)` plus a heap of free room ids (Meeting Rooms III).
- *Bookings arrive one by one?* → sorted starts plus `bisect` (My Calendar I), or the delta map for counts (My Calendar III).
- *Intersect two lists of intervals?* → two pointers; advance the one that ends first.
- *Fewest arrows, or keep the most intervals?* → sort by end.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Employee Free Time | `intervals/employee_free_time.py` | free time = the gaps between merged busy blocks; a heap merges the k sorted schedules in O(N log k) |
| Insert Interval | `intervals/insert_interval.py` | already sorted: copy what ends before, swallow what overlaps, copy the rest; O(n), no sort |
| Meeting Rooms II | `intervals/meeting_rooms_ii.py` · `practice/simple/49_meeting_rooms_ii.py` | sort by start; free ended rooms, push the end, count (or +1/−1 events, ends first); the peak is the answer |
| Merge Intervals | `intervals/merge_intervals.py` · `practice/simple/48_merge_intervals.py` | sort by start; only the last block can overlap; stretch its end with max |
| Minimum Interval to Include Each Query | `intervals/minimum_interval_to_include_each_query.py` | sort the queries offline; push started intervals by size; lazily pop the ended ones |
| My Calendar III | `intervals/my_calendar_iii.py` | difference map, +1 at start and −1 at end; the peak of the running sum is the answer |
| Non-overlapping Intervals | `intervals/non_overlapping_intervals.py` | keep the interval that ends first (exchange argument); count those starting before the kept end |
| Rectangle Area II | `intervals/rectangle_area_ii.py` | sweep x; each strip adds width × union length of the active y-intervals |

### Self-check

1. To keep the most non-overlapping intervals, why sort by *end*, and not by start or by length?
<details><summary>Answer</summary>The interval that ends first leaves the most room for everything after it, and any best answer can swap its first interval for that one without a clash. Start order can grab a long blocker such as <code>[1, 100]</code>. Length order fails too: in <code>[[1, 5], [4, 6], [5, 10]]</code> the shortest, <code>[4, 6]</code>, clashes with both others, while <code>[1, 5]</code> and <code>[5, 10]</code> fit together.</details>

2. In the rooms loop, why must `rooms = max(...)` come after both the freeing loop and the push?
<details><summary>Answer</summary>It reads how many meetings run at this instant. Before the freeing loop, meetings that already ended are still counted (<code>[1, 5]</code>, <code>[5, 9]</code> would need 2 rooms). Before the push, the newcomer is missing (<code>[[1, 5], [2, 6], [3, 7]]</code> would say 2 instead of 3).</details>

3. Write the overlap test for two closed intervals `a` and `b` without any `if`.
<details><summary>Answer</summary><code>a[0] &lt;= b[1] and b[0] &lt;= a[1]</code>. The common part is <code>[max(a[0], b[0]), min(a[1], b[1])]</code>, which is non-empty exactly when the test is true.</details>

4. Why does `merge` compare each interval with `merged[-1]` and not with the interval just before it?
<details><summary>Answer</summary>The block can reach further than the last interval: after <code>[1, 10]</code> and the nested <code>[2, 3]</code>, the previous end is 3 but the block ends at 10. Comparing with the previous interval would split <code>[4, 5]</code> off a block that already covers it.</details>

<a id="s15"></a>

## Greedy

> Make the move that looks best right now, never take it back, and carry only a tiny summary of the past: a running sum, the furthest reach, the fuel in the tank. Greedy is fast because it never branches; it is correct only when you can say why the greedy move never closes a door that the best answer needed.

Sorting by end in [Intervals & Sweep Line](#s14) was already a greedy: keep the interval that finishes first and never look back. This section asks when such a local choice is provably safe, and, when it is not, how a heap from [Heaps](#s13) lets you take a choice back.

**Reach for it when** one number can summarise everything that matters about the prefix you have seen (best sum ending here, furthest index reachable, fuel so far), the question asks for the minimum number of jumps / taps / patches / removals, sorting by one key (end time, deadline) makes "take the one that finishes first" obviously safe, or a small swap argument shows the locally best move is never worse. Typical words: "minimum number of …", "can you reach …", "maximum … subarray", "as many as possible".

**In this repo:** `greedy/` (14 problems) · greedy on intervals (Non-overlapping Intervals) is in [Intervals & Sweep Line](#s14); greedy with a heap (IPO, refuelling stops, Course Schedule III) is in [Heaps](#s13) · basics: `practice/simple/basics/graphs/03_union_find.py` (the cycle view of Couples Holding Hands), `practice/simple/basics/monotonic_stacks/05_remove_k_digits.py` (a greedy kept honest by a stack)

### The picture

Two walks carry the whole section. Maximum Subarray asks for the largest sum of a contiguous run, and Kadane's algorithm walks it with one running sum that it extends or drops. Jump Game II asks for the fewest jumps from the first index to the last, where `nums[i]` is the longest jump allowed from index i, and its walk grows levels of reachable indices.

```text
Kadane: extend, or restart when the past only hurts
nums:      -2    1   -3    4   -1    2    1   -5    4
cur:       -2    1   -2    4    3    5    6    1    5        cur = max(x, cur + x)
                 ^         ^              ^
              restart   restart        best = 6               (both restarts drop a carried -2)

Jump Game II: breadth-first search whose levels are intervals
nums      2  3  1  1  4
index     0  1  2  3  4
level 0  [0]               farthest = 0 + 2 = 2
level 1     [1  2]         farthest = max(1 + 3, 2 + 1) = 4
level 2           [3  4]   holds the last index: 2 jumps
```

The brute force tries every choice: every subarray, O(n²), every jump path, exponentially many, every subset of taps, 2ⁿ. Greedy notices that one number summarises everything about the past that the future cares about, so each step is a single `max` or a single comparison: O(n), or O(n log n) with a sort in front.

### Can I trust the greedy choice?

With greedy the code is the easy part; correctness is the risk. Two plain-language arguments cover almost every interview greedy, and one habit catches the rest.

The **exchange argument** says: swap in the greedy choice, and nothing gets worse. Take any best answer, find the first place where it differs from the greedy, and swap its choice for the greedy one; the answer stays valid and is no worse. Repeat, and the best answer turns into the greedy answer without ever getting worse. To keep the most non-overlapping intervals, a best answer's first interval can be swapped for the interval that *ends* first: it ends no later, so it can't clash with anything kept after it.

The second argument, **greedy stays ahead**, picks a measure of progress and shows that after k steps the greedy is never behind any other strategy. In Jump Game II, after j jumps the greedy's reach is the furthest that *any* j jumps can get, because it took the best reach from everything reachable in j − 1 jumps. Ahead at every step means first to the end.

The habit is the cross-check from [From Idea to Code](01_Start_Here.ipynb#s01): run the greedy against an exhaustive search on a few hundred tiny random inputs. A wrong greedy usually fails within seconds, and the failing input shows you *why*.

### When greedy fails, and what to do

Greedy fails when today's best move blocks a better future. Coin Change (322), the fewest coins that pay an amount, is the classic: with coins `[1, 3, 4]`, "biggest coin first" pays 6 as 4 + 1 + 1, three coins, instead of 3 + 3, two. Don't stare harder at the greedy: *test* it. The cell pits biggest-coin-first against a brute force that tries 0, 1, 2, ... coins, and prints the first amount where the two disagree, or `None`.

In [ ]:
from itertools import combinations_with_replacement


def coins_greedy(coins, amount):
    count = 0
    for c in sorted(coins, reverse=True):    # biggest coin first, as many as fit
        count += amount // c
        amount %= c
    return count if amount == 0 else -1


def coins_brute(coins, amount):              # fewest coins: try 0, 1, 2, ... coins
    for k in range(amount + 1):
        if any(sum(combo) == amount for combo in combinations_with_replacement(coins, k)):
            return k
    return -1


def first_counterexample(fast, slow, coins, amounts):
    for amount in amounts:
        if fast(coins, amount) != slow(coins, amount):
            return amount, fast(coins, amount), slow(coins, amount)
    return None


print(first_counterexample(coins_greedy, coins_brute, [1, 3, 4], range(1, 15)))       # (6, 3, 2)
print(first_counterexample(coins_greedy, coins_brute, [1, 5, 10, 25], range(1, 15)))  # None

**Try it**
- Predict, then run, the coins `[1, 7, 10]`: the first counterexample is `(14, 5, 2)`, greedy 10 + 1 + 1 + 1 + 1 against 7 + 7.
- Print `combo` when `coins_brute` succeeds for `[1, 3, 4]` and amount 6: it is `(3, 3)`, found at k = 2.
- Try a greedy of your own on Jump Game II: always jump as far as the current number allows. Write it in a few lines and run it on `[2, 3, 1, 1, 4]`: it lands on indices 2, 3 and 4, three jumps, while 0 → 1 → 4 takes two. The farthest landing spot is not the spot that reaches farthest.

When the test breaks the greedy, there are four ways forward.

1. **Try another greedy key.** Many greedies fail with one sort key and work with another: keeping the most intervals fails by start and works by end.
2. **Greedy with regret.** Some need a key *and* a way to undo. Course Schedule III, the most courses `[duration, last day]` that finish by their deadlines, fails both shortest-first (`[[3, 3], [1, 4]]`: 1 instead of 2) and deadline-first-if-it-fits (`[[5, 5], [4, 6], [2, 6]]`: 1 instead of 2). Deadline order plus a heap that drops the longest course taken works; it is in [Heaps](#s13), next to the refuelling stops.
3. **Search with pruning.** If choices really interact, explore them: backtracking with good pruning, in [Backtracking](08_Backtracking.ipynb#s16).
4. **Remember sub-answers.** If the same sub-question keeps coming back, such as "the fewest coins for amount a", that is [Dynamic Programming](08_Backtracking.ipynb#s25), and Coin Change is its textbook example.

### From idea to code

*Walk left to right carrying the smallest summary of the past that decides the future; at each item make the locally best update, usually one `max`, and record the answer the moment it is known.*

The **State** is one or two numbers that summarise the prefix, and their **Definition** is half the proof: `cur` is the best sum of a subarray that *ends* at i, and `farthest` is the furthest index reachable with one more jump. The **Invariant** is that the summary is exact for the prefix `0..i` and that the choices so far can still be completed into a best answer, which the exchange or stays-ahead argument proves. A **Step** folds item i in with one local choice, usually a `max`.

A **Fix** starts the next stretch at a boundary: a new jump level, a new candidate start when the tank runs dry, a new part when no letter reaches further. The **Record** comes right after the step, or at the boundary, where a jump is counted. The **Init** is the first item, `cur = best = nums[0]`, or a neutral `reach = 0`, never `best = 0` when every value can be negative. The **Return** is `best`, a count, or the failure value: `False` when `i > reach`, −1 when the total gas falls short of the total cost.

Each phrase of the idea then becomes one line, and their order is a decision too. "Best sum ending here: extend, or start fresh" is `cur = max(x, cur + x)`, and "best so far" is `best = max(best, cur)`, after it, because the step must first make `cur` the best sum ending *here*. "The furthest I can get" is `farthest = max(farthest, i + nums[i])`, and "this jump's range is used up" is `if i == cur_end:`, after it, because i belongs to the level that is closing.

The cell holds both templates. Maximum Subarray asks for the largest sum of a non-empty contiguous subarray: `[-2, 1, -3, 4, -1, 2, 1, -5, 4]` gives 6, from `[4, -1, 2, 1]`. Jump Game II gives the longest jump allowed from each index and asks for the fewest jumps from index 0 to the last index, which is always reachable: `[2, 3, 1, 1, 4]` takes 2, by way of index 1.

In [ ]:
def max_subarray(nums):
    cur = best = nums[0]                     # STATE + INIT: cur = best sum of a subarray ENDING here
    for x in nums[1:]:
        cur = max(x, cur + x)                # STEP: extend, or restart if the past only hurts
        best = max(best, cur)                # RECORD: cur is final for this index now
    return best                              # RETURN


def min_jumps(nums):
    jumps = cur_end = farthest = 0           # STATE + INIT: cur_end = last index reachable with `jumps` jumps;
                                             #   farthest = furthest index reachable with one more jump
    for i in range(len(nums) - 1):           # stop before the last index: arriving needs no jump
        farthest = max(farthest, i + nums[i])    # STEP: i is in the current level, fold in its reach
        if i == cur_end:                     # FIX: the level is used up; the next one ends at farthest
            jumps += 1                       # RECORD: one more jump
            cur_end = farthest
    return jumps                             # RETURN


print(max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4]), max_subarray([-3, -1, -2]))   # 6 -1
print(min_jumps([2, 3, 1, 1, 4]), min_jumps([0]))                                   # 2 0

**Try it**
- Start from `cur = best = 0` and loop over all of `nums`: `max_subarray([-3, -1, -2])` returns 0, a sum that no non-empty subarray has.
- Swap the two lines in `max_subarray`'s loop (record, then step): `max_subarray([1, 2])` returns 1. The last step is never recorded.
- In `min_jumps`, move the `if i == cur_end:` block above the `farthest = ...` line: `min_jumps([2, 3, 1, 1, 4])` returns 1. The first level closed before index 0's reach was counted.
- Loop `for i in range(len(nums))` (one step too far) and run `min_jumps([1, 1])`: 2 instead of 1. Standing on the last index triggered a jump to nowhere.

### Watch it work

The trace runs Kadane on the array from the picture, one line per number: the carried sum is extended or dropped, and `best` keeps the largest `cur`. A restart happens exactly when the carried sum is negative, because a negative past only lowers every sum that includes it.

In [ ]:
def trace_kadane(nums):
    cur = best = nums[0]
    print(f"x={nums[0]:>2}  cur={cur:>2}  best={best:>2}  (start)")
    for x in nums[1:]:
        note = "restart: the carried sum was negative" if cur < 0 else "extend"
        cur = max(x, cur + x)
        best = max(best, cur)
        print(f"x={x:>2}  cur={cur:>2}  best={best:>2}  {note}")


trace_kadane([-2, 1, -3, 4, -1, 2, 1, -5, 4])

**Try it**
- Run `trace_kadane([5, -9, 6])`: the 5 is carried into −4, then dropped, and `cur` restarts at 6, which is also the best.
- Run `trace_kadane([-3, -1, -2])`: every step restarts, and `best` ends as the single largest number, −1.
- Run `trace_kadane([2, -1, 2])`: the dip is worth crossing; `cur` goes 2, 1, 3.

### Where it goes wrong

1. **Starting `best` at 0, or restarting at 0.** Kadane on `[-3]` returns 0. Start from `nums[0]` and restart to `x`, not to 0.
2. **Checking reachability after updating it.** In Jump Game, which asks whether the last index can be reached at all, updating `reach` before testing `i > reach` lets you jump *from* an index you never reached: `[0, 1]` becomes `True`.
3. **One loop step too many.** Jump Game II stops at index n − 2; standing on the last index needs no jump, and `[1, 1]` gives 2 instead of 1.
4. **Updating `hi` before computing `lo`** in Maximum Product Subarray, the largest product of a contiguous run. Compute both from the *old* pair in one tuple assignment; the line-by-line version turns `[-1, -2, -1]` into 4, a product no subarray has, where the answer is 2.
5. **Overwriting instead of `max` in Candy's second pass.** Candy gives children in a row the fewest candies such that a child rated higher than a neighbour gets more. The right-to-left pass must keep what the left pass needed: `[1, 2, 3, 1]` gives 6 instead of 7.
6. **Restarting at `i` instead of `i + 1`** in Gas Station, which asks from which station a car can drive the whole circle. The station where the tank went negative belongs to the doomed stretch: `([1, 2, 3, 4, 5], [3, 4, 5, 1, 2])` returns 2 instead of 3.
7. **Dropping Gas Station's total check.** The scan only finds the last candidate; the total says whether any candidate works: `([2, 3, 4], [3, 4, 3])` returns 2 instead of −1.
8. **Trusting a greedy without a proof or a test.** "Biggest coin first" fails for coins `[1, 3, 4]` and amount 6.

### Edge cases to say out loud

One element · all negative (Kadane) · zeros (Maximum Product restarts there) · `[0]` (already at the end: reachable, 0 jumps) · a 0 you must cross (`[3, 2, 1, 0, 4]` is stuck) · total gas below total cost · equal neighbours (Candy: no rule between them) · ties in a sort order (by end, then start descending, for the two-points-per-interval problem at the end). The cell checks the cases that apply to the two templates.

In [ ]:
assert max_subarray([5]) == 5
assert max_subarray([-3, -1, -2]) == -1                  # all negative: never 0
assert max_subarray([0, -1, 0]) == 0
assert max_subarray([2, -1, 2]) == 3                     # a dip worth crossing
assert min_jumps([0]) == 0                               # already at the end
assert min_jumps([1, 1]) == 1
assert min_jumps([5, 1, 1, 1]) == 1                      # one jump covers everything
assert min_jumps([1, 2, 1, 1, 1]) == 3
print("edge cases pass")

**Try it**
- Predict `max_subarray([-1, 5, -1])` (5), then add the assert.
- What should `min_jumps([2, 1])` be? One jump: from index 0 you reach index 1 directly. Add it.
- Run `min_jumps([1, 0, 1])`: 2, because `min_jumps` assumes the end is reachable. Jump Game's `i > reach` test does not catch it, because the loop never stands on the last index. Check instead whether a closing level reached anything new: add `if farthest <= i: return -1` as the first line inside `if i == cur_end:`, as `min_taps` below does, and `[1, 0, 1]` and `[3, 2, 1, 0, 4]` both return −1.

### Variations

Every variation keeps the one-pass shape and changes what the summary holds: a second number, a range of states, a sort in front, or a second pass.

| Variation | What changes from the template | Problems |
|---|---|---|
| **Extend or restart** | the template; save `start` at each restart to return the subarray; circular: also cut out the most negative run | Maximum Subarray (53); Maximum Sum Circular Subarray (918): the same when a run may wrap around the end |
| **Signs flip the order** | carry the largest AND the smallest product ending here | Maximum Product Subarray (152) |
| **Reachable frontier** | one number `reach`; stuck when `i > reach` | Jump Game (55) |
| **Frontier in levels** | the template: count a jump (a clip) when `i == cur_end` | Jump Game II (45); Video Stitching (1024): the fewest clips that cover [0, T] |
| **Running sum with restart** | tank < 0 after i: no start in that stretch works, so restart at i + 1 | Gas Station (134) |
| **Intervals in disguise** | each letter spans [first, last]; cut where no span crosses | Partition Labels (763): the most parts with every letter in one part only |
| **Two passes** | satisfy the left neighbours, then the right ones; keep the max | Candy (135) |
| **Smallest first** | the smallest card left must start a run; use runs in sorted order | Hand of Straights (846): can the cards be split into runs of w consecutive values |
| **A range of possible states** | track the lowest and highest possible open-bracket count; `*` widens the range | Valid Parenthesis String (678): can each `*` be read as `(`, `)` or nothing so that the brackets balance |
| **Sort by a difference** | sort people by cost to A minus cost to B; the first half go to A | Two City Scheduling (1029): fly half of 2n people to city A and half to B at the least total cost |
| **Two running minimums** | keep the smallest value, and the smallest value with something smaller before it; a bigger third wins | Increasing Triplet Subsequence (334): is there i < j < k with nums[i] < nums[j] < nums[k] |
| *Second pass:* **cover a range** | bucket each tap by its left end, then the levels template; a level that reaches nothing new is a dry gap | Minimum Number of Taps to Open to Water a Garden (1326): the fewest taps that water a garden `[0, n]` |
| *Second pass:* **covered prefix** | sums 1..reach are covered; patch reach + 1 to double it | Patching Array (330): the fewest numbers to add so that every sum 1..n can be made |
| *Second pass:* **work backwards** | undo the last move first: it is the one still fully visible | Stamping the Sequence (936): the stamp presses that spell a target |
| *Second pass:* **rightmost points** | sort by end; new points go at the right end, where they serve the most later intervals | Set Intersection Size At Least Two (757): the fewest points that hit every interval twice |
| *Second pass:* **bounds that meet** | the answer is the largest lower bound, and it is always reachable | Super Washing Machines (517): the fewest moves that even out the loads; Minimum Number of Increments on Subarrays to Form a Target Array (1526): the fewest +1 strokes that build a target |
| *Second pass:* **swap into place** | fix the couches left to right; each swap seats one couple, and a k-cycle needs k − 1 | Couples Holding Hands (765): the fewest swaps that seat every couple together |

The first variation answers the two follow-ups interviewers attach to Kadane. To return the subarray and not only its sum, remember where the current run started and copy `(start, i)` whenever the best improves: `[-2, 1, -3, 4, -1, 2, 1, -5, 4]` gives the sum 6 on indices 3 to 6. Maximum Sum Circular Subarray lets a run wrap around the end, so `[5, -3, 5]` gives 10, the run 5, 5.

The best circular run either does not wrap, which is plain Kadane, or wraps around the end. A wrapping run is everything except one middle run, so cutting out the *most negative* run gives `total − worst`. When every number is negative nothing may be cut, and the answer is the plain Kadane best.

In [ ]:
def max_subarray_span(nums):
    cur, start = nums[0], 0                  # STATE: cur = best sum ending at i; start = where that run starts
    best, l, r = nums[0], 0, 0
    for i in range(1, len(nums)):
        if cur < 0:                          # STEP: restart here ...
            cur, start = nums[i], i
        else:                                # ... or extend
            cur += nums[i]
        if cur > best:                       # RECORD: remember where the best run starts and ends
            best, l, r = cur, start, i
    return best, l, r                        # RETURN


def max_circular_subarray(nums):             # 918
    best = cur_hi = cur_lo = worst = nums[0]
    for x in nums[1:]:
        cur_hi = max(x, cur_hi + x)          # STEP: Kadane for the largest run ...
        best = max(best, cur_hi)
        cur_lo = min(x, cur_lo + x)          # ... and for the most negative run
        worst = min(worst, cur_lo)
    return best if best < 0 else max(best, sum(nums) - worst)   # RETURN: all negative, nothing to wrap


print(max_subarray_span([-2, 1, -3, 4, -1, 2, 1, -5, 4]))           # (6, 3, 6)
print(max_circular_subarray([5, -3, 5]), max_circular_subarray([-3, -2, -3]))   # 10 -2

**Try it**
- Change `cur > best` to `cur >= best` and run `max_subarray_span([3, -3, 3])`: the sum stays 3, but the span moves from `(0, 0)` to `(0, 2)`. Ties pick a different subarray that is also correct, so ask which one is wanted.
- Drop the `best < 0` guard and run `max_circular_subarray([-3, -2, -3])`: 0, the sum of an empty "everything except the whole array".
- Print `sum(nums) - worst` for `[5, -3, 5]`: 7 − (−3) = 10, the run 5, 5 that wraps around the end.

A negative number turns the largest product into the smallest and the smallest into the largest, and that is the next variation. Maximum Product Subarray asks for the largest product of a contiguous run: `[2, 3, -2, 4]` gives 6 and `[-2, 3, -4]` gives 24. So carry both the largest and the smallest product ending here: whichever one the next negative flips into the lead is already in hand. A zero resets both to 0, a fresh start.

In [ ]:
def max_product(nums):
    hi = lo = best = nums[0]                 # STATE + INIT: hi / lo = largest / smallest product ENDING here
    for x in nums[1:]:
        hi, lo = max(x, hi * x, lo * x), min(x, hi * x, lo * x)   # STEP: both from the OLD hi and lo
        best = max(best, hi)                 # RECORD
    return best                              # RETURN


print(max_product([2, 3, -2, 4]), max_product([-2, 0, -1]), max_product([-2, 3, -4]))   # 6 0 24

**Try it**
- Split the tuple assignment into two lines (`hi = ...`, then `lo = ...`): `max_product([-1, -2, -1])` gives 4, a product no subarray has, where the answer is 2. `lo` was computed from the *new* `hi`.
- Print `hi, lo` after each step for `[-2, 3, -4]`: `(3, -6)`, then `(24, -12)`. The −6 waited in `lo` until −4 flipped it into the lead.
- Keep only `hi` (plain Kadane with products, `hi = max(x, hi * x)`): `[-2, 3, -4]` gives 3 instead of 24.

Jump Game drops the counting and asks only whether the last index can be reached, with `nums[i]` the longest jump from index i: `[2, 3, 1, 1, 4]` gives True, and `[3, 2, 1, 0, 4]` gives False, because every path gets stuck on the 0 at index 3. The indices you can reach always form a prefix `[0, reach]`, so one number is the whole state. Check before you step: an index beyond `reach` can't be stood on, so its jump must not count.

In [ ]:
def can_jump(nums):
    reach = 0                                # STATE: furthest index reachable from indices 0..i
    for i, step in enumerate(nums):
        if i > reach:                        # RETURN: I can't even stand here
            return False
        reach = max(reach, i + step)         # STEP: standing on i, fold in its jump
    return True                              # RETURN


print(can_jump([2, 3, 1, 1, 4]), can_jump([3, 2, 1, 0, 4]), can_jump([0]))   # True False True

**Try it**
- Move the `reach = ...` line above the `if`: `can_jump([0, 1])` becomes `True`. You jumped from index 1 without ever standing on it.
- Print `i, reach` for `[3, 2, 1, 0, 4]`: reach is 3 at indices 0 to 3, and index 4 is beyond it.
- Predict, then run: `can_jump([1, 0, 1])` is `False`, because index 2 lies past reach 1, and `can_jump([2, 0, 0])` is `True`, because the last index is exactly the reach.

Gas Station moves the running sum onto a circle. Station i gives `gas[i]`, and the drive to the next station costs `cost[i]`; the question is from which station a car with an empty tank can drive the whole circle, or −1: `gas = [1, 2, 3, 4, 5]` and `cost = [3, 4, 5, 1, 2]` give 3.

If you start at s and the tank first goes negative after station i, then no start between s and i works either: you reached each of them with a non-negative tank, so starting there with an empty one is no better. Jump the candidate start straight to i + 1. If the total gas covers the total cost, the last candidate is the answer, because it sits right after the lowest point of the running balance.

```text
gas - cost   -2  -2  -2   3   3
balance      -2  -4  -6  -3   0      the lowest point is after station 2
start at 3, right after the lowest point: the tank there is balance + 6, never negative
before the end, and total >= 0 keeps it non-negative after wrapping round (tanks 3, 6, 4, 2, 0)
```

The code keeps both balances in one pass: `total` over the whole circle decides whether any start works, and `tank` since the candidate start decides where the candidate moves. The fix is the restart, which moves the candidate to i + 1 and empties the tank the moment `tank` drops below 0.

In [ ]:
def can_complete_circuit(gas, cost):
    total = tank = start = 0                 # STATE: total = balance of the whole loop;
                                             #   tank = balance since the candidate start
    for i in range(len(gas)):
        diff = gas[i] - cost[i]
        total += diff                        # STEP: station i joins both balances
        tank += diff
        if tank < 0:                         # FIX: dry after i, so no start in start..i works:
            start, tank = i + 1, 0           #   restart at i + 1 with an empty tank
    return start if total >= 0 else -1       # RETURN


print(can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]), can_complete_circuit([2, 3, 4], [3, 4, 3]))   # 3 -1

**Try it**
- Restart at `i` instead of `i + 1` (`start, tank = i, 0`): the first call returns 2, and starting at station 2 dies at once, with 3 gas against a cost of 5.
- Print `start, tank` at the end of each step of the first call: the start jumps to 1, 2, 3 while the tank keeps dying, then the tank climbs to 3 and 6.
- Return `start` without the total check: the second call returns 2 instead of −1. The scan finds a candidate even when no start works.

Partition Labels cuts a string into as many parts as possible so that each letter appears in one part only, and returns the part sizes: `"ababcbacadefegdehijhklij"` gives `[9, 7, 8]`. Every letter spans from its first to its last copy, and a part must swallow the whole span of every letter it touches.

```text
s:    a b a b c b a c a | d e f e g d e | h i j h k l i j
a:    [---------------]
b:      [-------]
c:            [-----]
d:                        [---------]
e:                          [---------]       cut where i == end: after 8, 15, 23 -> [9, 7, 8]
```

So sweep left to right and push the part's end out to `last[c]` for every letter met. When `i` catches up with the end, no letter inside reaches further, and the part closes.

In [ ]:
def partition_labels(s):
    last = {c: i for i, c in enumerate(s)}   # STATE: each letter's span ends at its last copy
    sizes, start, end = [], 0, 0             # STATE: the current part is s[start..end] so far
    for i, c in enumerate(s):
        end = max(end, last[c])              # STEP: the part must reach every letter it has touched
        if i == end:                         # FIX: nothing inside reaches further, close the part
            sizes.append(end - start + 1)    # RECORD
            start = i + 1
    return sizes                             # RETURN


print(partition_labels("ababcbacadefegdehijhklij"))   # [9, 7, 8]

**Try it**
- Test `i == end` *before* updating `end`: `partition_labels("abab")` gives `[1, 3]` instead of `[4]`. The first `a` was cut off before its span was known.
- Predict, then run: `partition_labels("abc")` is `[1, 1, 1]` and `partition_labels("abca")` is `[4]`.
- Print `i, end` for `"abab"`: `end` jumps to 2 at the first `a` and to 3 at the first `b`, and the part closes only at index 3.

Candy hands out candies to children in a row: everyone gets at least one, and a child rated higher than a neighbour gets more than that neighbour; the question is the fewest candies in total, so `[1, 0, 2]` needs 5, as 2, 1, 2. Each child has two rules, one per neighbour. One left-to-right pass satisfies every left rule with the fewest candies, and one right-to-left pass does the same for the right rules.

```text
ratings:      1   2   3   1   0
left pass:    1   2   3   1   1      climb from the left
right pass:   1   1   3   2   1      climb from the right
max of both:  1   2   3   2   1      = 9 candies
```

A child must satisfy both rules, so it takes the larger of its two counts. That is why the second pass keeps a `max` instead of overwriting.

In [ ]:
def candy(ratings):
    n = len(ratings)
    c = [1] * n                              # STATE + INIT: candies; everyone starts with 1
    for i in range(1, n):                    # pass 1, left -> right
        if ratings[i] > ratings[i - 1]:
            c[i] = c[i - 1] + 1              # STEP: beat a lower left neighbour
    for i in range(n - 2, -1, -1):           # pass 2, right -> left
        if ratings[i] > ratings[i + 1]:
            c[i] = max(c[i], c[i + 1] + 1)   # FIX: beat a lower right neighbour, keep what pass 1 needed
    return sum(c)                            # RETURN


print(candy([1, 0, 2]), candy([1, 2, 2]), candy([1, 2, 3, 1, 0]))   # 5 4 9

**Try it**
- Replace `max(c[i], c[i + 1] + 1)` with `c[i + 1] + 1`: `candy([1, 2, 3, 1])` gives 6 instead of 7. The peak lost the 3 it needed for its left side.
- Print `c` after each pass for `[1, 2, 3, 1, 0]` and compare with the picture.
- Run `candy([2, 2, 2])`: 3. Equal neighbours have no rule between them, so everyone gets 1.

Hand of Straights asks whether a hand of cards can be split into runs of `w` consecutive values: `[1, 2, 3, 6, 2, 3, 4, 7, 8]` with `w = 3` can, as 1 2 3, 2 3 4 and 6 7 8. Look at the smallest card left: it can't sit in the middle of a run, because that run would need a smaller card. So every copy of it *starts* a run, which forces how many copies of the next `w − 1` values are used. Repeat with the next smallest card.

In [ ]:
def is_n_straight_hand(hand, w):             # 846
    count = Counter(hand)                    # STATE: copies of each card still unused
    for x in sorted(count):                  # smallest card first
        need = count[x]                      # every x left must START a run: nothing smaller is left
        if need:
            for y in range(x, x + w):        # the runs need `need` copies of x, x+1, ..., x+w-1
                if count[y] < need:
                    return False             # RETURN: a run can't be completed
                count[y] -= need             # STEP: use them up
    return True                              # RETURN


print(is_n_straight_hand([1, 2, 3, 6, 2, 3, 4, 7, 8], 3), is_n_straight_hand([1, 2, 3, 4, 5], 4))   # True False

**Try it**
- Iterate `for x in count` (input order) instead of `sorted(count)`: `is_n_straight_hand([2, 1, 3], 3)` gives `False` instead of `True`. The 2 tried to start a run although the 1 was still waiting.
- Print `x, need` inside the `if need:` for the first call: runs start at 1, 2 and 6, once each.
- Keep the print and run `is_n_straight_hand([1, 1, 2, 2, 3, 3], 3)`: `True`, with need = 2 at x = 1, two runs starting at once. Then run `is_n_straight_hand([1, 2, 3], 2)`: `False`, because the run that 3 must start needs a 4.

The rest of this section is a second pass: Hard problems that reuse the same moves. Skip them until the main path is automatic.

Minimum Number of Taps to Open to Water a Garden has a garden `[0, n]` with a tap at every integer point, tap i watering `[i − ranges[i], i + ranges[i]]`, and asks for the fewest taps that water the whole garden, or −1: n = 5 with ranges `[3, 4, 1, 1, 0, 0]` needs 1, the tap at 1, which waters `[−3, 5]`.

It is Jump Game II in disguise. Bucket every tap by its left end, so that `right_end[l]` is the furthest right end of a tap that starts at `l`, then count the "jumps" needed to cover `[0, n]`. A level that reaches nothing new is a dry gap, and the answer is −1.

In [ ]:
def min_taps(n, ranges):
    right_end = [0] * (n + 1)                # STATE: right_end[l] = furthest right end of a tap starting at l
    for i, r in enumerate(ranges):
        left = max(0, i - r)
        right_end[left] = max(right_end[left], i + r)
    taps = cur_end = farthest = 0            # exactly min_jumps, with right_end[i] for i + nums[i]
    for i in range(n):
        farthest = max(farthest, right_end[i])   # STEP
        if i == cur_end:                     # FIX: the watered prefix ends here
            if farthest <= i:                # nothing waters past i: a dry gap
                return -1                    # RETURN
            taps += 1                        # RECORD: open one more tap
            cur_end = farthest
    return taps                              # RETURN


print(min_taps(5, [3, 4, 1, 1, 0, 0]), min_taps(3, [0, 0, 0, 0]), min_taps(7, [1, 2, 1, 0, 2, 1, 0, 1]))   # 1 -1 3

**Try it**
- Drop the clamp (`left = i - r`) and rerun the first call: −1 instead of 1. `right_end[-3]` silently wrote into index 3, because Python reads a negative index from the end, so no tap seems to start at 0.
- Run `min_taps(0, [5])`: 0. A garden that is a single point needs no tap, and the loop never runs.
- Delete the dry-gap check and run `min_taps(3, [0, 0, 0, 0])`: 1 instead of −1. A level that reached nothing new was still counted as a tap.
- Print `right_end` for the third call: `[3, 3, 6, 3, 6, 0, 8, 0]`. Read as Jump Game II's `i + nums[i]`, it gives three levels, ending at 3, 6 and 8: three taps.

Patching Array gives a sorted list and n, and asks for the fewest numbers to add so that every value in 1..n is a sum of some of the numbers: `[1, 3]` with n = 6 needs one patch, the 2. Keep `reach` with the promise that every value in 1..reach is a subset sum.

A number x ≤ reach + 1 glues on and extends the promise to 1..reach + x. A bigger x leaves reach + 1 as a hole that no later, bigger number can fill, so patch reach + 1 itself: it doubles the coverage, the most any single number can add.

In [ ]:
def min_patches(nums, n):
    reach, patches, i = 0, 0, 0              # STATE: every value in 1..reach is a subset sum
    while reach < n:
        if i < len(nums) and nums[i] <= reach + 1:   # STEP: glues on, now 1..reach + nums[i]
            reach += nums[i]
            i += 1
        else:                                # FIX: reach + 1 is a hole, patch it (coverage doubles)
            reach += reach + 1
            patches += 1                     # RECORD
    return patches                           # RETURN


print(min_patches([1, 3], 6), min_patches([1, 5, 10], 20), min_patches([1, 2, 2], 5))   # 1 2 0

**Try it**
- Change `nums[i] <= reach + 1` to `nums[i] <= reach`: `min_patches([1, 3], 6)` gives 2 instead of 1. The first patch is a 1, although the array already had one.
- Print `reach` after each patch of `min_patches([], 20)`: 1, 3, 7, 15, 31. With no numbers at all, patching 1, 2, 4, 8, 16 doubles the coverage each time: 5 patches.
- Patch a smaller value instead, say always 1 (`reach += 1`): `min_patches([], 20)` needs 20 patches instead of 5.

Five more Hard greedies are worth knowing by their idea alone, and their code sits in the repo files listed in the Problem map.

Stamping the Sequence presses a stamp onto a row of `?` until it spells the target, each press overwriting the letters under it, and asks for the order of presses: stamp `"abc"` builds `"ababc"` by pressing at 0, then at 2. Work backwards, because the last press is still fully visible: peel any window that matches the stamp, with `?` matching anything, turn its letters into `?`, and reverse the peel order at the end.

Set Intersection Size At Least Two asks for the smallest set of integers that holds at least two points of every closed interval: `[[1, 3], [3, 7], [8, 9]]` needs 5, such as {2, 3, 7, 8, 9}. Sort by end, equal ends by start descending so that the narrower interval comes first, and give an interval that lacks points the rightmost ones it can take, e, or e − 1 and e when both are missing, because points at the right end serve the most later intervals; only the two largest points chosen so far ever matter.

Super Washing Machines lets any set of machines each pass one dress to a neighbour in one move, and asks for the fewest moves that even out the loads, or −1: `[1, 0, 5]` takes 3. The answer is the larger of two lower bounds, and it is always reached: the dresses that must cross a boundary, the absolute prefix balance, and an overloaded machine's own excess, because it hands out one dress per move.

Minimum Number of Increments on Subarrays to Form a Target Array builds a target from zeros with strokes that add 1 to a whole subarray, and asks for the fewest strokes: `[3, 1, 5, 4, 2, 3, 4, 2]` takes 9. Every stroke has one left edge, and a step up of d needs d new left edges, so the answer is `target[0]` plus the sum of the positive steps up.

Couples Holding Hands seats couples (0, 1), (2, 3), ... in a row of two-seat couches and asks for the fewest swaps of two people that put every couple side by side: `[0, 2, 1, 3]` needs 1. Fix the couches from left to right, swapping each partner in; with couples as nodes and couches as edges the seating splits into cycles, a cycle of k couples needs exactly k − 1 swaps, and this fixing never wastes one, so the answer is couples − cycles.

### Say it in the interview

> "Brute force tries every choice: O(n²) subarrays, or exponentially many paths and subsets. The greedy move is ___, and it is safe because ___: either an exchange (any best answer can swap its first choice for mine and stay valid and no worse) or stays-ahead (after every step I am at least as far as any other plan). So one pass carrying ___ decides everything: O(n), or O(n log n) with a sort in front."

Fill in the blanks out loud. Jump Game II: "the move is to make the next level end at the farthest index any index in this level can reach; it is safe because after j jumps no plan reaches further; one pass carrying `cur_end` and `farthest`, O(n)." Likely follow-ups and your answers:

- *Return the subarray, not the sum* → save `start` when you restart; copy `(start, i)` whenever `best` improves.
- *The array is circular* (918) → `max(best, total − most negative run)`, unless every number is negative.
- *Jump Game II, but the end may be unreachable* → when a level closes, `if farthest <= i: return -1`.
- *Gas Station: why is `total >= 0` enough?* → the start right after the lowest point of the running balance never runs dry (the picture above).
- *Prove it* → one exchange or stays-ahead sentence.
- *Your greedy breaks on a counterexample* → a regret heap from [Heaps](#s13), or search, or [Dynamic Programming](08_Backtracking.ipynb#s25).

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Candy | `greedy/candy.py` | left pass beats the left neighbour, right pass the right one; each child takes the max of both |
| Couples Holding Hands | `greedy/couples_holding_hands.py` | couples are nodes, couches are edges: swaps = couples − cycles; fixing couches left to right achieves it |
| Gas Station | `greedy/gas_station.py` | if the tank dies after i, no start up to i works: restart at i + 1; possible iff total ≥ 0 |
| Jump Game | `greedy/jump_game.py` | reachable indices form a prefix: track reach, fail as soon as i > reach |
| Jump Game II | `greedy/jump_game_ii.py` | BFS whose levels are intervals: jump when i reaches cur_end, then cur_end = farthest |
| Maximum Product Subarray | `greedy/maximum_product_subarray.py` | a negative swaps the max and the min: carry both products ending here |
| Maximum Subarray | `greedy/maximum_subarray.py` | Kadane: cur = max(x, cur + x); a negative running sum is dead weight |
| Minimum Number of Increments on Subarrays to Form a Target Array | `greedy/min_number_operations.py` | every stroke has one left edge: target[0] + the sum of the positive steps up |
| Minimum Number of Taps to Open to Water a Garden | `greedy/minimum_number_of_taps_to_open_to_water_a_garden.py` | bucket taps by left end, then Jump Game II over the right ends; −1 when a level reaches nothing new |
| Partition Labels | `greedy/partition_labels.py` | each letter spans [first, last]; push end to last[c], cut when i == end |
| Patching Array | `greedy/patching_array.py` | sums 1..reach are covered; x ≤ reach + 1 extends it, otherwise patch reach + 1 (doubling) |
| Set Intersection Size At Least Two | `greedy/set_intersection_size_at_least_two.py` | sort by end (ties: start descending); keep the two largest points; add missing ones at e − 1, e |
| Stamping The Sequence | `greedy/stamping_the_sequence.py` | work backwards: peel any window matching the stamp ('?' is a wildcard), then reverse the order |
| Super Washing Machines | `greedy/super_washing_machines.py` | two lower bounds, abs(prefix balance) and a machine's own excess; the larger one is always reachable |

### Self-check

1. In Kadane, why is it safe to throw away a negative running sum?
<details><summary>Answer</summary>Any subarray that ends later and keeps the negative carried part would be strictly larger without it. So the best subarray ending at the next index is either that number alone or an extension of a non-negative carry: dropping a negative prefix never loses the optimum.</details>

2. In Hand of Straights, why must every copy of the smallest card left start a run?
<details><summary>Answer</summary>A run is w consecutive values. If the smallest card left sat in the middle or at the end of a run, that run would need a smaller card, and none is left. So all its copies start runs, and that decides how many copies of the next w − 1 values are used up.</details>

3. Why does Candy's second pass use `max` instead of overwriting?
<details><summary>Answer</summary>The left pass already set the smallest count that satisfies the left neighbour. Overwriting with the right-pass value can lower it and break that rule: in <code>[1, 2, 3, 1]</code> the peak needs 3 because of its left side but only 2 because of its right side.</details>

4. In an interview, how do you check a greedy you are not sure about?
<details><summary>Answer</summary>Attack it with tiny inputs first: ties, zeros, negatives, one huge item, and the input where a "bigger now" choice blocks the future. If it survives, state the exchange or stays-ahead argument in one sentence. When practising, run it against a brute force on a few hundred random small inputs.</details>